In [1]:
#!/usr/bin/env python3
"""
KITTI→V-KITTI Inference Script - UPDATED FOR NATIVE RESOLUTION MODEL
Ultra-precise ControlNet inference using newly trained k2vk_baseline_native model (1248×376 resolution)
to generate V-KITTI style images from KITTI inputs.

Model: /training/vkitti_seg/checkpoints_trained/checkpoints/k2vk_baseline_native/
Trained: 2025-10-02, 22000 steps, final loss: 0.21636 (latest finetune)
Resolution: 1248×376 (native KITTI dimensions)

=== TERMINOLOGY NOTE (Paper Alignment) ===
- DFF = Decisive Feature Fidelity (paper terminology)
- Previously called "RFF" in legacy code - now renamed for consistency
- DFF measures mechanism parity via counterfactual explanations
"""


import os
import random
import gc
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
from safetensors.torch import load_file


# === DATASET PATHS ===
KITTI_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/data/data_for_new_mvp/KITTI"
VKITTI_SEG_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/data/data_for_new_mvp/class_segmentation_v_kitti"
# Use the latest finetuned baseline (22K steps)
BASELINE_MODEL_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_01_baseline"
# Newly trained variants (current workspace outputs)
OVF_STEERING_MODEL_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_02_ovf_steering_beta0p70"
OVF_YOLOP_DA_MODEL_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_03_ovf_yolop_da_beta0p50"
OVF_YOLOP_LL_MODEL_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_04_ovf_yolop_ll_beta0p45"
# DFF (Decisive Feature Fidelity) Counterfactual models - renamed from RFF for paper consistency
DFF_YOLOP_LL_CF_MODEL_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_07_dff_yolop_ll_beta0p30_lam0p08"
DFF_YOLOP_DA_CF_MODEL_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_06_dff_yolop_da_beta0p30_lam0p08"
DFF_STEERING_CF_MODEL_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_05_dff_steering_beta0p35_lam0p10"


# === V-KITTI CLASS MAPPING ===
VKITTI_CLASS_MAPPING = {
    'Terrain': (210, 0, 200, 0), 'Sky': (90, 200, 255, 1), 'Tree': (0, 199, 0, 2),
    'Vegetation': (90, 240, 0, 3), 'Building': (140, 140, 140, 4), 'Road': (100, 60, 100, 5),
    'GuardRail': (250, 100, 255, 6), 'TrafficSign': (255, 255, 0, 7), 'TrafficLight': (200, 200, 0, 8),
    'Pole': (255, 130, 0, 9), 'Misc': (80, 80, 80, 10), 'Truck': (160, 60, 60, 11),
    'Car': (255, 127, 80, 12), 'Van': (0, 139, 139, 13), 'Undefined': (0, 0, 0, 14)
}


# Create RGB to index lookup
RGB_TO_INDEX = {(r, g, b): idx for _, (r, g, b, idx) in VKITTI_CLASS_MAPPING.items()}


# Local fallback for objective labels if previous cell hasn't been run
try:
    objective_for
except NameError:
    _OBJECTIVE_MAP_LOCAL = {
        "Baseline": "reconstruction",
        "OVF-Steering": "reconstruction + OVF (steering)",
        "OVF-YOLOP-DA": "reconstruction + OVF (drivable area)",
        "OVF-YOLOP-LL": "reconstruction + OVF (lane lines)",
        # DFF = Decisive Feature Fidelity (paper terminology, was RFF in legacy code)
        "DFF-YOLOP-LL-CF": "reconstruction + OVF (lane lines) + DFF counterfactual",
        "DFF-YOLOP-DA-CF": "reconstruction + OVF (drivable area) + DFF counterfactual",
        "DFF-Steering-CF": "reconstruction + OVF (steering) + DFF counterfactual",
    }
    def objective_for(model_name: str) -> str:
        if model_name in _OBJECTIVE_MAP_LOCAL:
            return _OBJECTIVE_MAP_LOCAL[model_name]
        if "DFF" in model_name.upper() or "RFF" in model_name.upper():
            return "reconstruction + OVF + DFF"
        return "unknown objective"


# Quiet flag hookup
QUIET = globals().get('QUIET', True)

def load_controlnet_pipeline(model_path: str = BASELINE_MODEL_PATH):
    """Load trained ControlNet pipeline with optional quiet mode."""
    try:
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model not found at {model_path}")
        with suppress_output(QUIET):
            controlnet = ControlNetModel.from_pretrained(
                model_path,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                use_safetensors=True
            )
            pipeline = StableDiffusionXLControlNetPipeline.from_pretrained(
                "stabilityai/stable-diffusion-xl-base-1.0",
                controlnet=controlnet,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                variant="fp16" if torch.cuda.is_available() else None,
                use_safetensors=True
            )
            try:
                pipeline.set_progress_bar_config(disable=True)
            except Exception:
                pass
            pipeline.enable_model_cpu_offload()
            if hasattr(pipeline, 'enable_xformers_memory_efficient_attention'):
                try:
                    pipeline.enable_xformers_memory_efficient_attention()
                except Exception:
                    pass
        return pipeline
    except Exception as e:
        if not QUIET:
            print(f"❌ Error loading pipeline: {e}")
        return None

def generate_vkitti_proper(pipeline, pure_conditioning, content_hints, *, steps=80, guidance=8.0, cond_scale=0.8, seed=42):
    """Generate V-KITTI style image using PROPER architecture with optional quiet mode."""
    try:
        base_prompt = "Transform real KITTI driving scene to V-KITTI virtual style"
        content_description = ", ".join(content_hints) if content_hints else "standard scene"
        enhanced_prompt = f"{base_prompt}, {content_description}, photorealistic V-KITTI synthetic rendering"
        negative_prompt = "blurry, low quality, distorted, cartoon, painting, sketch, artifacts, noise, real photography, KITTI style"
        with suppress_output(QUIET):
            generated_image = pipeline(
                prompt=enhanced_prompt,
                negative_prompt=negative_prompt,
                image=pure_conditioning,
                num_inference_steps=steps,
                guidance_scale=guidance,
                controlnet_conditioning_scale=cond_scale,
                generator=torch.Generator().manual_seed(seed)
            ).images[0]
        return generated_image
    except Exception as e:
        if not QUIET:
            print(f"❌ Error during generation: {e}")
        from PIL import Image
        placeholder = Image.new('RGB', (1248, 376), (0, 0, 0))
        return placeholder

2026-01-21 19:09:12.308124: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-21 19:09:12.375930: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/networkhome/WMGDS/safaei_d/miniconda3/envs/img2img/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid 

In [2]:
# Define GENERATORS for the seven trained models by scanning your checkpoints folder
# NOTE: Generator names use DFF (Decisive Feature Fidelity) per paper terminology
import os

CHECKPOINTS_DIR = r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7"

def _has_model_files(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    cfg = os.path.join(path, 'config.json')
    has_cfg = os.path.isfile(cfg)
    # accept common weight filenames
    weights = [
        'diffusion_pytorch_model.safetensors',
        'pytorch_model.bin',
        'model.safetensors',
    ]
    has_wt = any(os.path.isfile(os.path.join(path, w)) for w in weights)
    return has_cfg and has_wt

def _pick_first(*candidates):
    for c in candidates:
        if c and _has_model_files(c):
            return c
    return None

baseline_dir = _pick_first(
    os.path.join(CHECKPOINTS_DIR, 'out_01_baseline'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_baseline_native'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_vkitti_seg_baseline'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_vkitti_seg_baseline_clean'),
)

ovf_steer_dir = _pick_first(
    os.path.join(CHECKPOINTS_DIR, 'out_02_ovf_steering_beta0p70'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_ovf_steering_corrected_final'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_ovf_steering_corrected'),
)

ovf_yolop_da_dir = _pick_first(
    os.path.join(CHECKPOINTS_DIR, 'out_03_ovf_yolop_da_beta0p50'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_ovf_yolop_da_corrected'),
)

ovf_yolop_ll_dir = _pick_first(
    os.path.join(CHECKPOINTS_DIR, 'out_04_ovf_yolop_ll_beta0p45'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_ovf_yolop_ll_corrected'),
)

# DFF = Decisive Feature Fidelity (paper terminology, checkpoint dirs still use legacy 'rff' naming)
dff_steer_dir = _pick_first(
    os.path.join(CHECKPOINTS_DIR, 'out_05_dff_steering_beta0p35_lam0p10'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_rff_steering_counterfactual'),
)

dff_yolop_da_cf_dir = _pick_first(
    os.path.join(CHECKPOINTS_DIR, 'out_06_dff_yolop_da_beta0p30_lam0p08'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_rff_yolop_da_counterfactual'),
)

dff_yolop_ll_cf_dir = _pick_first(
    os.path.join(CHECKPOINTS_DIR, 'out_07_dff_yolop_ll_beta0p30_lam0p08'),
    os.path.join(CHECKPOINTS_DIR, 'k2vk_rff_yolop_ll_counterfactual'),
)

_candidates = {
    'Baseline': baseline_dir,
    'OVF-Steering': ovf_steer_dir,
    'OVF-YOLOP-DA': ovf_yolop_da_dir,
    'OVF-YOLOP-LL': ovf_yolop_ll_dir,
    # DFF generators (Decisive Feature Fidelity - paper terminology)
    'DFF-Steering-CF': dff_steer_dir,
    'DFF-YOLOP-DA-CF': dff_yolop_da_cf_dir,
    'DFF-YOLOP-LL-CF': dff_yolop_ll_cf_dir,
}

# Keep only those that passed the file check
GENERATORS = {k: v for k, v in _candidates.items() if v is not None}

if not QUIET:
    print("Configured GENERATORS (found models):")
    for k, v in GENERATORS.items():
        print(f" - {k}: {v}")
    _missing = [k for k, v in _candidates.items() if v is None]
    if _missing:
        print("\nWarning: Missing models (no model files found):")
        for k in _missing:
            print(" -", k)

if not GENERATORS:
    raise RuntimeError("No valid generator checkpoints found. Please verify checkpoints directory and filenames.")

In [3]:
# =============================================================================
# RQ1: Evaluate all 7 trained generators with robust metrics and pairing
# =============================================================================
# CRITICAL FIXES APPLIED (per paper audit):
# 1. RFF → DFF naming (Decisive Feature Fidelity, paper terminology)
# 2. Fixed counterfactual target formula (now seeks minimal perturbation)
# 3. Clarified metric semantics (similarity vs loss vs distance)
# 4. Added tolerant metrics for sparse lane lines (Chamfer + F1)
# 5. Compute percentile-based thresholds instead of hardcoded values
# 6. FIXED: Counterfactual now ALWAYS returns a result (no more None)
# 7. STABILITY: Disabled torch.compile by default to prevent autograd/SigmoidBackward0 errors

import os, sys, gc, math, random, re, inspect, importlib.util, contextlib
from typing import Tuple, Optional, Callable
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm

# Fix TF32 warnings and ensure precision settings
try:
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

try:
    from skimage.metrics import structural_similarity as ssim
except Exception:
    ssim = None

try:
    import lpips
except Exception:
    lpips = None

from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel

# Quiet helpers
def _maybe_tqdm(iterable, desc=None, total=None):
    if globals().get('QUIET', False):
        return iterable
    return tqdm(iterable, desc=desc, total=total)

# === User-provided SUT paths (steering + YOLOP) ===
STEERING_ARCH_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/models/architectures/For_steering_angle_prediction_tweaked_v3.py"
STEERING_WEIGHTS_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/models/weights/final_model.pth"
YOLOP_ROOT = r"/networkhome/WMGDS/safaei_d/3_generators/models/pretrained/YOLOP"
YOLOP_WEIGHTS_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/models/pretrained/YOLOP/weights/End-to-end.pth"

# Fallbacks for globals from previous cells
if 'KITTI_PATH' not in globals():
    KITTI_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/data/data_for_new_mvp/KITTI"
if 'VKITTI_SEG_PATH' not in globals():
    VKITTI_SEG_PATH = r"/networkhome/WMGDS/safaei_d/3_generators/data/data_for_new_mvp/class_segmentation_v_kitti"
if 'GENERATORS' not in globals():
    raise RuntimeError("GENERATORS dict is not defined. Please run the configuration cell that defines GENERATORS.")
if 'USE_GPU' not in globals():
    USE_GPU = torch.cuda.is_available()
if 'QUIET' not in globals():
    QUIET = True

# === CRITICAL: Define Splits per Paper ===
SCENE_SPLITS = {
    'calibration': ['Scene01', 'Scene02', 'Scene06'],  # 80% of data
    'heldout':     ['Scene18', 'Scene20']              # 20% of data (Test set)
}

# Map KITTI seq to V-KITTI scene based on your workspace
SEQ2SCENE = {
    '0001': 'Scene01',
    '0002': 'Scene02',
    '0006': 'Scene06',
    '0018': 'Scene18',
    '0020': 'Scene20',
}

# Helper fallbacks if not defined previously
if 'prepare_proper_conditioning' not in globals():
    def prepare_proper_conditioning(kitti_image_path, seg_path, target_size=(1248, 376)):
        kitti_img = Image.open(kitti_image_path).convert('RGB')
        kitti_resized = kitti_img.resize(target_size, Image.LANCZOS)
        seg_img = Image.open(seg_path).convert('RGB')
        seg_resized = seg_img.resize(target_size, Image.NEAREST)
        return seg_resized, kitti_resized, seg_resized, seg_img.size, []

_frame_re = re.compile(r"^(\d{6})\.png$")
_seg_re = re.compile(r"^classgt_(\d{5})\.png$")

def find_matching_kitti_seg_pair(split: str = 'heldout') -> Tuple[str, str, str, int]:
    """
    Finds a random pair from the specified split.
    split: 'calibration' (for threshold finding) or 'heldout' (for final evaluation)
    """
    allowed_scenes = set(SCENE_SPLITS[split])
    valid_seqs = [s for s in SEQ2SCENE if SEQ2SCENE[s] in allowed_scenes]
    available_seqs = [d for d in os.listdir(KITTI_PATH) if os.path.isdir(os.path.join(KITTI_PATH, d))]
    candidates = [s for s in valid_seqs if s in available_seqs]
    
    if not candidates:
        raise RuntimeError(f"No sequences found for split '{split}' (Scenes: {allowed_scenes})")
    
    random_seq = random.choice(candidates)
    kseq = os.path.join(KITTI_PATH, random_seq)
    scene = SEQ2SCENE[random_seq]
    seg_dir = os.path.join(VKITTI_SEG_PATH, scene)
    if not os.path.isdir(seg_dir):
        raise RuntimeError(f"Missing V-KITTI scene folder: {seg_dir}")
    kitti_frames = {int(m.group(1)) for f in os.listdir(kseq) if (m := _frame_re.match(f))}
    seg_frames = {int(m.group(1)) for f in os.listdir(seg_dir) if (m := _seg_re.match(f))}
    common = sorted(kitti_frames.intersection(seg_frames))
    if not common:
        raise RuntimeError(f"No common frames between KITTI/{random_seq} and {scene}")
    frame_num = random.choice(common)
    kitti_image_path = os.path.join(kseq, f"{frame_num:06d}.png")
    seg_path = os.path.join(seg_dir, f"classgt_{frame_num:05d}.png")
    return kitti_image_path, seg_path, random_seq, frame_num

# Cache LPIPS once
_LPIPS_MODEL = None

def _get_lpips():
    global _LPIPS_MODEL
    if lpips is None:
        return None
    device = 'cuda' if USE_GPU else 'cpu'
    if _LPIPS_MODEL is None:
        try:
            model = lpips.LPIPS(net='alex')
            model = model.to(device)
            model.eval()
            for p in model.parameters():
                p.requires_grad_(False)
            _LPIPS_MODEL = model
        except Exception:
            _LPIPS_MODEL = None
    else:
        try:
            param_device = next(_LPIPS_MODEL.parameters()).device
        except StopIteration:
            param_device = torch.device(device)
        desired = torch.device(device)
        if param_device != desired:
            _LPIPS_MODEL = _LPIPS_MODEL.to(desired)
    return _LPIPS_MODEL

# --------------------------- Generators ---------------------------

# =============================================================================
# OPTIMIZATION: Cached Base Pipeline
# Instead of loading full SDXL base model for each generator (slow ~8-15s),
# we cache the base pipeline and only swap the ControlNet weights (~0.5-1s)
# =============================================================================
_CACHED_BASE_PIPE = None
_CACHED_BASE_DTYPE = None

def _get_cached_base_pipe(dtype):
    """Get or create cached SDXL base pipeline for ControlNet swapping."""
    global _CACHED_BASE_PIPE, _CACHED_BASE_DTYPE
    
    if _CACHED_BASE_PIPE is not None and _CACHED_BASE_DTYPE == dtype:
        return _CACHED_BASE_PIPE
    
    # Clear old cache if dtype changed
    if _CACHED_BASE_PIPE is not None:
        del _CACHED_BASE_PIPE
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    with suppress_output(globals().get('QUIET', False)):
        base_id = "stabilityai/stable-diffusion-xl-base-1.0"
        # Create a minimal placeholder ControlNet for initialization
        # This will be replaced with actual weights later
        placeholder_config = {
            "in_channels": 4,
            "conditioning_channels": 3,
            "flip_sin_to_cos": True,
            "freq_shift": 0,
        }
        try:
            # Try loading a minimal controlnet config from first available checkpoint
            first_ckpt = list(GENERATORS.values())[0] if GENERATORS else None
            if first_ckpt and os.path.isdir(first_ckpt):
                placeholder_cn = ControlNetModel.from_pretrained(first_ckpt, torch_dtype=dtype, use_safetensors=True)
            else:
                placeholder_cn = None
        except Exception:
            placeholder_cn = None
        
        if placeholder_cn is not None:
            _CACHED_BASE_PIPE = StableDiffusionXLControlNetPipeline.from_pretrained(
                base_id,
                controlnet=placeholder_cn,
                torch_dtype=dtype,
                variant="fp16" if dtype == torch.float16 else None,
                use_safetensors=True,
            )
        else:
            # Fallback: load without controlnet first
            from diffusers import StableDiffusionXLPipeline
            base_pipe = StableDiffusionXLPipeline.from_pretrained(
                base_id, torch_dtype=dtype, variant="fp16" if dtype == torch.float16 else None, use_safetensors=True)
            _CACHED_BASE_PIPE = base_pipe
    
    _CACHED_BASE_DTYPE = dtype
    
    try:
        _CACHED_BASE_PIPE.set_progress_bar_config(disable=True)
    except Exception:
        pass
    try:
        _CACHED_BASE_PIPE.enable_model_cpu_offload()
    except Exception:
        pass
    if hasattr(_CACHED_BASE_PIPE, 'enable_xformers_memory_efficient_attention') and torch.cuda.is_available():
        try:
            _CACHED_BASE_PIPE.enable_xformers_memory_efficient_attention()
        except Exception:
            pass
    
    return _CACHED_BASE_PIPE


def load_trained_pipeline(model_path: str):
    """
    Load ControlNet pipeline with OPTIMIZED caching.
    
    Instead of loading the full SDXL base model each time (~8-15s),
    we cache the base pipeline and only swap ControlNet weights (~0.5-1s).
    This provides 5-10x speedup on model switching.
    """
    if not os.path.isdir(model_path):
        raise FileNotFoundError(f"Generator path not found: {model_path}")
    
    use_fp16 = torch.cuda.is_available()
    dtype = torch.float16 if use_fp16 else torch.float32
    
    with suppress_output(globals().get('QUIET', False)):
        # Load new ControlNet weights
        controlnet = ControlNetModel.from_pretrained(model_path, torch_dtype=dtype, use_safetensors=True)
        
        # Get cached base pipeline
        base_pipe = _get_cached_base_pipe(dtype)
        
        # Create new pipeline with swapped ControlNet
        # This is much faster than loading from scratch
        pipe = StableDiffusionXLControlNetPipeline(
            vae=base_pipe.vae,
            text_encoder=base_pipe.text_encoder,
            text_encoder_2=base_pipe.text_encoder_2,
            tokenizer=base_pipe.tokenizer,
            tokenizer_2=base_pipe.tokenizer_2,
            unet=base_pipe.unet,
            controlnet=controlnet,
            scheduler=base_pipe.scheduler,
        )
        try:
            pipe.set_progress_bar_config(disable=True)
        except Exception:
            pass
    
    try:
        pipe.enable_model_cpu_offload()
    except Exception:
        pass
    if hasattr(pipe, 'enable_xformers_memory_efficient_attention') and torch.cuda.is_available():
        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception:
            pass
    return pipe

# --------------------------- SUTs (match eval.ipynb) ---------------------------
# OPTIMIZATION: torch.compile for 1.5-2x faster SUT inference
# Caches compiled models to avoid recompilation overhead

_COMPILED_YOLOP_MODEL = None
_COMPILED_STEERING_MODEL = None

def _ensure_yolop_paths():
    if YOLOP_ROOT not in sys.path:
        sys.path.append(YOLOP_ROOT)
    lib_path = os.path.join(YOLOP_ROOT, 'lib')
    if os.path.isdir(lib_path) and lib_path not in sys.path:
        sys.path.append(lib_path)

def load_yolop_sut(use_compile: bool = False) -> Tuple[Optional[torch.nn.Module], Optional[Callable[[Image.Image], torch.Tensor]]]:
    """Load YOLOP model with optional torch.compile optimization.
    
    Args:
        use_compile: If True, applies torch.compile for 1.5-2x faster inference.
                     DEFAULT FALSE: Enabling this causes SigmoidBackward0 errors during saliency checks.
    """
    global _COMPILED_YOLOP_MODEL
    
    _ensure_yolop_paths()
    try:
        from lib.models.YOLOP import get_net
        model = get_net(False)
    except Exception:
        try:
            from models.YOLOP import get_net
            model = get_net(False)
        except Exception as exc:
            raise RuntimeError(f"CRITICAL: Failed to import YOLOP get_net: {exc}. Cannot proceed with placeholder.")
    
    if not os.path.exists(YOLOP_WEIGHTS_PATH):
        raise RuntimeError(f"CRITICAL: YOLOP weights not found at {YOLOP_WEIGHTS_PATH}")
    
    ckpt = torch.load(YOLOP_WEIGHTS_PATH, map_location="cpu")
    state = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state, strict=False)
    device = torch.device("cuda" if USE_GPU else "cpu")
    model = model.to(device).eval()
    
    # OPTIMIZATION: torch.compile for faster inference
    # DISABLED BY DEFAULT due to autograd instability
    if use_compile and torch.cuda.is_available() and hasattr(torch, 'compile'):
        if _COMPILED_YOLOP_MODEL is None:
            try:
                _COMPILED_YOLOP_MODEL = torch.compile(model, mode='reduce-overhead')
                if not globals().get('QUIET', False):
                    print("  ✓ YOLOP model compiled with torch.compile (1.5-2x speedup)")
            except Exception as e:
                if not globals().get('QUIET', False):
                    print(f"  ⚠ torch.compile failed for YOLOP: {e}, using eager mode")
                _COMPILED_YOLOP_MODEL = model
        model = _COMPILED_YOLOP_MODEL
    elif not globals().get('QUIET', False):
        print("  ℹ YOLOP torch.compile disabled for stability (prevents SigmoidBackward0 errors)")

    def preprocess(img_pil: Image.Image) -> torch.Tensor:
        arr = np.array(img_pil.resize((640, 384), Image.LANCZOS)).astype(np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)
        return tensor

    return model, preprocess

def load_steering_sut(
    arch_path: str = STEERING_ARCH_PATH,
    weights_path: str = STEERING_WEIGHTS_PATH,
    use_compile: bool = False,
) -> Tuple[Optional[torch.nn.Module], Optional[Callable[[Image.Image], torch.Tensor]]]:
    """Load steering model with optional torch.compile optimization.
    
    Args:
        arch_path: Path to steering architecture file
        weights_path: Path to steering weights
        use_compile: If True, applies torch.compile for 1.5-2x faster inference.
                     DEFAULT FALSE: Enabling this causes autograd instability.
    """
    global _COMPILED_STEERING_MODEL
    
    if not os.path.exists(arch_path):
        raise RuntimeError(f"CRITICAL: Steering architecture not found at {arch_path}")
    if not os.path.exists(weights_path):
        raise RuntimeError(f"CRITICAL: Steering weights not found at {weights_path}")
    
    spec = importlib.util.spec_from_file_location("steering_arch", arch_path)
    if spec is None or spec.loader is None:
        raise RuntimeError("CRITICAL: Failed to load steering architecture module spec")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    ModelClass = (
        getattr(module, "AutonomousDrivingModel", None)
        or getattr(module, "SimpleSteeringModel", None)
        or getattr(module, "SteeringModel", None)
    )
    if ModelClass is None:
        for _, obj in inspect.getmembers(module):
            if inspect.isclass(obj) and issubclass(obj, torch.nn.Module):
                ModelClass = obj
                break
    if ModelClass is None:
        raise RuntimeError("CRITICAL: No steering model class discovered in architecture file")
    
    model = ModelClass()
    state = torch.load(weights_path, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        model.load_state_dict(state["state_dict"], strict=False)
    else:
        model.load_state_dict(state, strict=False)
    device = torch.device("cuda" if USE_GPU else "cpu")
    model = model.to(device).eval()
    
    # OPTIMIZATION: torch.compile for faster inference
    # DISABLED BY DEFAULT due to autograd instability
    if use_compile and torch.cuda.is_available() and hasattr(torch, 'compile'):
        if _COMPILED_STEERING_MODEL is None:
            try:
                _COMPILED_STEERING_MODEL = torch.compile(model, mode='reduce-overhead')
                if not globals().get('QUIET', False):
                    print("  ✓ Steering model compiled with torch.compile (1.5-2x speedup)")
            except Exception as e:
                if not globals().get('QUIET', False):
                    print(f"  ⚠ torch.compile failed for Steering: {e}, using eager mode")
                _COMPILED_STEERING_MODEL = model
        model = _COMPILED_STEERING_MODEL
    elif not globals().get('QUIET', False):
        print("  ℹ Steering torch.compile disabled for stability")

    def preprocess(img_pil: Image.Image) -> torch.Tensor:
        arr = np.array(img_pil.resize((200, 66), Image.LANCZOS)).astype(np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)
        return tensor

    return model, preprocess

def to_prob_mask(seg_output: torch.Tensor) -> torch.Tensor:
    if not isinstance(seg_output, torch.Tensor):
        seg_output = torch.as_tensor(seg_output)
    if seg_output.dim() == 4:
        b, c, _, _ = seg_output.shape
        if c == 2:
            probs = torch.softmax(seg_output, dim=1)[:, 1]
        elif c == 1:
            probs = torch.sigmoid(seg_output)[:, 0]
        else:
            probs = torch.softmax(seg_output, dim=1).max(dim=1).values
    elif seg_output.dim() == 3:
        probs = torch.sigmoid(seg_output)
    else:
        probs = torch.sigmoid(seg_output.unsqueeze(0))
    return probs.clamp(0, 1)

def _first_tensor(obj):
    if isinstance(obj, torch.Tensor):
        return obj
    if isinstance(obj, (list, tuple)):
        for item in obj:
            tensor = _first_tensor(item)
            if tensor is not None:
                return tensor
    if isinstance(obj, dict):
        for item in obj.values():
            tensor = _first_tensor(item)
            if tensor is not None:
                return tensor
    return None

def _pick_spatial_tensor(obj):
    candidates = []
    def collect(val):
        if isinstance(val, torch.Tensor) and val.dim() >= 3:
            candidates.append(val)
        elif isinstance(val, (list, tuple)):
            for elem in val:
                collect(elem)
        elif isinstance(val, dict):
            for elem in val.values():
                collect(elem)
    collect(obj)
    if not candidates:
        return _first_tensor(obj)
    candidates.sort(
        key=lambda t: int(t.shape[-1]) * int(t.shape[-2]) if t.dim() >= 3 else 0,
        reverse=True,
    )
    return candidates[0]

def yolop_select_da_head(out):
    candidate = out
    if isinstance(out, (list, tuple) ) and len(out) > 1:
        candidate = out[1]
    tensor = _pick_spatial_tensor(candidate)
    if tensor is None:
        tensor = _pick_spatial_tensor(out)
    return tensor

def yolop_select_ll_head(out):
    candidate = out
    if isinstance(out, (list, tuple)):
        candidate = out[2] if len(out) > 2 else out[-1]
    tensor = _pick_spatial_tensor(candidate)
    if tensor is None:
        tensor = _pick_spatial_tensor(out)
    return tensor

def _yolop_prob_mean(head_tensor) -> torch.Tensor:
    tensor = head_tensor if isinstance(head_tensor, torch.Tensor) else _first_tensor(head_tensor)
    if tensor is None:
        raise RuntimeError("YOLOP head is not a tensor")
    probs = to_prob_mask(tensor)
    return probs.mean()

# =============================================================================
# INPUT FIDELITY (IV) - LPIPS→SSIM→PSNR fallback
# =============================================================================

def compute_input_fidelity(img_real: Image.Image, img_gen: Image.Image) -> Optional[float]:
    """
    Compute Input-Value (IV) fidelity. 
    Paper Section V-B: IV Score = 1 - LPIPS_norm ∈ [0,1], where LPIPS clamped to [0,1].
    Returns: float in [0,1], higher is better.
    """
    try:
        w, h = img_real.size
        img_gen_resized = img_gen.resize((w, h), Image.LANCZOS)
        arr_real = np.array(img_real).astype(np.float32) / 255.0
        arr_gen = np.array(img_gen_resized).astype(np.float32) / 255.0
        lp = _get_lpips()
        if lp is not None:
            t1 = torch.from_numpy(arr_real).permute(2, 0, 1).unsqueeze(0)
            t2 = torch.from_numpy(arr_gen).permute(2, 0, 1).unsqueeze(0)
            device = next(lp.parameters()).device if hasattr(lp, "parameters") else torch.device("cpu")
            t1 = t1.to(device)
            t2 = t2.to(device)
            distance = float(lp(t1, t2).item())
            # Paper: IV Score = 1 - LPIPS_norm (clamped)
            return max(0.0, min(1.0, 1.0 - distance))
        if ssim is not None:
            score = float(ssim(arr_real, arr_gen, data_range=1.0, channel_axis=2))
            return max(0.0, min(1.0, score))
        mse = float(np.mean((arr_real - arr_gen) ** 2))
        if mse <= 1e-8:
            return 1.0
        psnr = 10 * math.log10(1.0 / mse)
        return max(0.0, min(1.0, psnr / 50.0))
    except Exception:
        return None

# =============================================================================
# OUTPUT-VALUE FIDELITY (OV) - Similarity scores (higher is better)
# =============================================================================

def compute_ovf_steering_similarity(model, preprocess, img_real: Image.Image, img_gen: Image.Image) -> Optional[float]:
    """
    Compute OV Fidelity SIMILARITY for steering model.
    
    Paper Section V-B formula: exp(-5 · |θr - θs|) ∈ [0,1]
    Implementation: exp(-diff/0.2) = exp(-5·diff), mathematically equivalent.
    
    Returns: float in [0, 1] where 1 = identical outputs (HIGHER is better)
    """
    if model is None or preprocess is None:
        return None
    try:
        with torch.no_grad():
            real_tensor = preprocess(img_real)
            gen_tensor = preprocess(img_gen)
            y_real = model(real_tensor).view(-1).mean()
            y_gen = model(gen_tensor).view(-1).mean()
            diff = float(torch.abs(y_real - y_gen).item())
        # Paper formula: exp(-5 · |θr - θs|)
        # Here: 1/STEERING_SCALE = 1/0.2 = 5, so exp(-diff/0.2) = exp(-5·diff)
        STEERING_SCALE = 0.2  # = 1/5, matching paper's coefficient
        normalized_diff = diff / STEERING_SCALE
        score = math.exp(-normalized_diff)
        return max(0.0, min(1.0, score))
    except Exception as e:
        if not QUIET:
            print(f"[compute_ovf_steering_similarity] Error: {e}")
        return None

# Alias for backward compatibility
compute_ovf_steering = compute_ovf_steering_similarity

def _binarize(mask: np.ndarray, thresh: float = 0.5) -> np.ndarray:
    return (mask >= thresh).astype(np.uint8)

def _iou(a: np.ndarray, b: np.ndarray) -> Optional[float]:
    try:
        inter = np.logical_and(a, b).sum()
        union = np.logical_or(a, b).sum()
        if union == 0:
            return 1.0 if inter == 0 else 0.0
        return float(inter) / float(union)
    except Exception:
        return None

def _chamfer_distance_score(a: np.ndarray, b: np.ndarray) -> Optional[float]:
    """Chamfer Distance Score for sparse masks. Returns [0,1], 1=perfect match."""
    try:
        from scipy.spatial.distance import cdist
        pts_a = np.argwhere(a > 0.5)
        pts_b = np.argwhere(b > 0.5)
        if len(pts_a) == 0 and len(pts_b) == 0:
            return 1.0
        if len(pts_a) == 0 or len(pts_b) == 0:
            return 0.0
        dists_a_to_b = cdist(pts_a, pts_b, metric='euclidean').min(axis=1)
        dists_b_to_a = cdist(pts_b, pts_a, metric='euclidean').min(axis=1)
        chamfer = (dists_a_to_b.mean() + dists_b_to_a.mean()) / 2.0
        score = np.exp(-chamfer / 10.0)
        return float(np.clip(score, 0.0, 1.0))
    except Exception:
        return None

def _f1_with_tolerance(a: np.ndarray, b: np.ndarray, tolerance_px: int = 3) -> Optional[float]:
    """F1 Score with spatial tolerance for sparse masks."""
    try:
        from scipy import ndimage
        if a.sum() == 0 and b.sum() == 0:
            return 1.0
        if a.sum() == 0 or b.sum() == 0:
            return 0.0
        struct = ndimage.generate_binary_structure(2, 2)
        dilated_a = ndimage.binary_dilation(a > 0.5, struct, iterations=tolerance_px)
        dilated_b = ndimage.binary_dilation(b > 0.5, struct, iterations=tolerance_px)
        tp_a = np.logical_and(a > 0.5, dilated_b).sum()
        tp_b = np.logical_and(b > 0.5, dilated_a).sum()
        precision = tp_a / max(1, (a > 0.5).sum())
        recall = tp_b / max(1, (b > 0.5).sum())
        if precision + recall < 1e-8:
            return 0.0
        f1 = 2 * precision * recall / (precision + recall)
        return float(np.clip(f1, 0.0, 1.0))
    except Exception:
        return None

def compute_ovf_yolop_similarity(model, preprocess, img_real: Image.Image, img_gen: Image.Image, 
                                  head: str = "da", use_tolerant_metric: bool = True) -> Optional[float]:
    """
    Compute OV Fidelity SIMILARITY for YOLOP model.
    
    Paper Section V-B: For YOLOP heads we use IoU between predicted masks (higher is better).
    For LL: Uses Chamfer + F1-tolerant hybrid (sparse mask - IoU inappropriate)
    
    Returns: float in [0, 1] where 1 = identical outputs (HIGHER is better)
    """
    if model is None or preprocess is None:
        return None
    try:
        with torch.no_grad():
            real_tensor = preprocess(img_real)
            gen_tensor = preprocess(img_gen)
            out_real = model(real_tensor)
            out_gen = model(gen_tensor)
            if head == "da":
                tensor_real = yolop_select_da_head(out_real)
                tensor_gen = yolop_select_da_head(out_gen)
            else:
                tensor_real = yolop_select_ll_head(out_real)
                tensor_gen = yolop_select_ll_head(out_gen)
            prob_real = to_prob_mask(tensor_real).squeeze().detach().cpu().numpy()
            prob_gen = to_prob_mask(tensor_gen).squeeze().detach().cpu().numpy()
        
        mask_real = _binarize(prob_real, 0.5)
        mask_gen = _binarize(prob_gen, 0.5)
        
        # For lane lines, use tolerant metrics (IoU inappropriate for sparse masks)
        if head == "ll" and use_tolerant_metric:
            chamfer_score = _chamfer_distance_score(mask_real, mask_gen)
            f1_score = _f1_with_tolerance(mask_real, mask_gen, tolerance_px=3)
            if chamfer_score is not None and f1_score is not None:
                return 0.4 * chamfer_score + 0.6 * f1_score
            elif f1_score is not None:
                return f1_score
            elif chamfer_score is not None:
                return chamfer_score
        
        return _iou(mask_real, mask_gen)
    except Exception as e:
        if not QUIET:
            print(f"[compute_ovf_yolop_similarity] Error: {e}")
        return None

# Alias for backward compatibility
compute_ovf_yolop = compute_ovf_yolop_similarity

# =============================================================================
# OUTPUT-VALUE FIDELITY LOSS (lower is better) - for training objectives
# =============================================================================

def compute_ovf_loss_steering(model, preprocess, img_real: Image.Image, img_gen: Image.Image, 
                               *, mode: str = 'mse') -> Optional[float]:
    """Compute OVF LOSS for steering (LOWER is better). Different from similarity!"""
    if model is None or preprocess is None:
        return None
    try:
        with torch.no_grad():
            real_tensor = preprocess(img_real)
            gen_tensor = preprocess(img_gen)
            y_real = model(real_tensor).view(-1).mean()
            y_gen = model(gen_tensor).view(-1).mean()
            if mode == 'mse':
                loss = float(F.mse_loss(y_real, y_gen).item())
            elif mode == 'l1':
                loss = float(F.l1_loss(y_real, y_gen).item())
            else:
                loss = float(torch.abs(y_real - y_gen).item())
        return loss
    except Exception:
        return None

def compute_ovf_loss_yolop(model, preprocess, img_real: Image.Image, img_gen: Image.Image, 
                           *, head: str = 'da', mode: str = 'mse') -> Optional[float]:
    """Compute OVF LOSS for YOLOP (LOWER is better). Different from similarity!"""
    if model is None or preprocess is None:
        return None
    try:
        with torch.no_grad():
            real_tensor = preprocess(img_real)
            gen_tensor = preprocess(img_gen)
            out_real = model(real_tensor)
            out_gen = model(gen_tensor)
            if head == 'da':
                tensor_real = yolop_select_da_head(out_real)
                tensor_gen = yolop_select_da_head(out_gen)
            else:
                tensor_real = yolop_select_ll_head(out_real)
                tensor_gen = yolop_select_ll_head(out_gen)
            prob_real = to_prob_mask(tensor_real).squeeze()
            prob_gen = to_prob_mask(tensor_gen).squeeze()
            if mode == 'mse':
                loss = float(F.mse_loss(prob_real, prob_gen).item())
            elif mode == 'l1':
                loss = float(F.l1_loss(prob_real, prob_gen).item())
            else:
                loss = float(F.binary_cross_entropy(prob_gen.clamp(1e-7, 1-1e-7), prob_real).item())
        return loss
    except Exception:
        return None

# =============================================================================
# DFF (Decisive Feature Fidelity) - Paper terminology (was RFF in legacy code)
# =============================================================================
# DFF measures mechanism parity via counterfactual explanations
# - DFF_similarity: cosine similarity of saliency maps (HIGHER is better)
# - DFF_distance: 1 - similarity (LOWER is better) - use for threshold checks
# - DFF_loss: MSE of counterfactual change maps (LOWER is better) - use for training

def _saliency(model, preprocess, img: Image.Image, output_selector=None) -> Optional[np.ndarray]:
    """Compute gradient-based saliency map for an image."""
    try:
        x = preprocess(img)
        x = x.requires_grad_(True)
        y = model(x)
        y_sel = output_selector(y) if output_selector is not None else (y.float().mean() if torch.is_tensor(y) else None)
        if y_sel is None:
            return None
        if not torch.is_tensor(y_sel):
            y_sel = torch.as_tensor(y_sel, device=x.device, dtype=x.dtype)
        y_sel = y_sel.mean()
        model.zero_grad(set_to_none=True)
        if x.grad is not None:
            x.grad.zero_()
        y_sel.backward()
        grad = x.grad.detach().float().abs().mean(dim=1, keepdim=False).squeeze(0)
        grad = grad.cpu().numpy()
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
        return grad
    except Exception:
        return None

def compute_dff_similarity(model, preprocess, img_real: Image.Image, img_gen: Image.Image, 
                           output_selector=None) -> Optional[float]:
    """
    Compute DFF (Decisive Feature Fidelity) as cosine SIMILARITY between saliency maps.
    Returns: float in [0, 1] where 1 = identical reasoning (HIGHER is better)
    """
    grad_real = _saliency(model, preprocess, img_real, output_selector)
    grad_gen = _saliency(model, preprocess, img_gen, output_selector)
    if grad_real is None or grad_gen is None:
        return None
    try:
        v1 = grad_real.reshape(-1)
        v2 = grad_gen.reshape(-1)
        num = float((v1 * v2).sum())
        den = float(np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
        return max(0.0, min(1.0, num / den))
    except Exception:
        return None

# Legacy alias (DEPRECATED - use compute_dff_similarity instead)
# Will be removed in future versions
compute_rff_similarity = compute_dff_similarity

def compute_dff_distance(model, preprocess, img_real: Image.Image, img_gen: Image.Image, 
                         output_selector=None) -> Optional[float]:
    """
    Compute DFF (Decisive Feature Fidelity) as DISTANCE.
    Returns: float in [0, 1] where 0 = identical reasoning (LOWER is better)
    Use this for threshold-based evaluation: DFF_distance <= epsilon indicates success.
    """
    similarity = compute_dff_similarity(model, preprocess, img_real, img_gen, output_selector)
    if similarity is None:
        return None
    return 1.0 - similarity
# Legacy alias (DEPRECATED - use compute_dff_distance instead)
# Will be removed in future versions
compute_rff_distance = compute_dff_distance

# =============================================================================
# COUNTERFACTUAL ROUTINES - COMPLETELY REWRITTEN FOR ROBUSTNESS
# =============================================================================
# CRITICAL FIX: Counterfactual now ALWAYS returns a result (no more None)
# The old implementation would return None if the flip threshold wasn't achieved.
# This caused DFF_loss to be None for many samples, breaking downstream analysis.
#
# NEW DESIGN:
# 1. Always track the BEST perturbation found (one with maximum output change)
# 2. If threshold is achieved, prefer minimal perturbation among successes
# 3. If threshold is NOT achieved, return the best effort (maximum change found)
# 4. Never return None unless there's an actual exception

def _tv_loss(x: torch.Tensor, weight: float = 1e-3) -> torch.Tensor:
    """Total variation loss for smoothness (sparsity constraint)."""
    dx = x[:, :, :, 1:] - x[:, :, :, :-1]
    dy = x[:, :, 1:, :] - x[:, :, :-1, :]
    return weight * (dx.abs().mean() + dy.abs().mean())

def _clamp01(x: torch.Tensor) -> torch.Tensor:
    return x.clamp(0.0, 1.0)

def _steering_counterfactual_robust(model, x_in: torch.Tensor, steps: int = 80, 
                                     lr: float = 3e-2, tv: float = 1e-3,
                                     target_flip_threshold: float = 0.1) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    ROBUST counterfactual for steering model - ALWAYS returns a result.
    
    OPTIMIZED: Eliminated duplicate model forward passes in tracking loop.
    Now reuses forward pass result from loss computation for tracking.
    
    Returns the perturbation that causes the maximum output change.
    If threshold is achieved, returns minimal perturbation among successes.
    
    Paper Algorithm 1: K=80 CF steps for decisive map generation.
    """
    device = x_in.device
    x_cf = x_in.detach().clone().requires_grad_(True)
    
    with torch.no_grad():
        y_orig = model(x_in).detach().mean()
    
    # Target: flip direction by threshold
    target_delta = target_flip_threshold * (-torch.sign(y_orig) if y_orig.abs() > 1e-6 else torch.tensor(1.0, device=device))
    target = y_orig + target_delta
    
    opt = torch.optim.Adam([x_cf], lr=lr)
    
    # Track TWO things:
    # 1. Best success (achieved threshold with minimal perturbation)
    # 2. Best effort (maximum output change regardless of threshold)
    best_success_cf = None
    best_success_diff = float('inf')
    best_effort_cf = x_cf.detach().clone()
    best_effort_change = 0.0
    
    for step in range(steps):
        opt.zero_grad()
        y_pred = model(x_cf)
        y_pred_mean = y_pred.mean()
        
        # OPTIMIZATION: Track BEFORE opt.step() to reuse forward pass result
        # This correctly associates output with the current x_cf state
        with torch.no_grad():
            current_change = (y_pred_mean.detach() - y_orig).abs().item()
            current_diff = (x_cf - x_in).abs().mean().item()
            
            # Track best effort (maximum change achieved)
            if current_change > best_effort_change:
                best_effort_change = current_change
                best_effort_cf = x_cf.detach().clone()
            
            # Track best success (achieved threshold with minimal perturbation)
            if current_change >= target_flip_threshold * 0.8:
                if current_diff < best_success_diff:
                    best_success_diff = current_diff
                    best_success_cf = x_cf.detach().clone()
        
        # Loss: encourage output to reach target + minimize perturbation + smoothness
        output_loss = F.mse_loss(y_pred_mean, target)
        perturbation_loss = (x_cf - x_in).abs().mean()
        tv_loss_val = _tv_loss(x_cf, tv)
        
        # Adaptive weighting: reduce perturbation penalty as we get close to target
        output_weight = 1.0
        perturb_weight = 0.1 if step < steps // 2 else 0.5
        
        loss = output_weight * output_loss + perturb_weight * perturbation_loss + tv_loss_val
        loss.backward()
        opt.step()
        
        with torch.no_grad():
            x_cf.data = _clamp01(x_cf.data)
    
    # Return best success if threshold achieved, otherwise best effort
    if best_success_cf is not None:
        result_cf = best_success_cf
    else:
        result_cf = best_effort_cf
    
    with torch.no_grad():
        diff = (result_cf - x_in).abs().mean(dim=1, keepdim=True)
    
    return result_cf, diff

# Keep old name for compatibility but use robust version
def _steering_counterfactual_minimal(model, x_in: torch.Tensor, steps: int = 80, 
                                      lr: float = 3e-2, tv: float = 1e-3,
                                      target_flip_threshold: float = 0.1) -> Tuple[torch.Tensor, torch.Tensor]:
    """Wrapper that calls robust version."""
    return _steering_counterfactual_robust(model, x_in, steps=steps, lr=lr, tv=tv, 
                                           target_flip_threshold=target_flip_threshold)

def _steering_counterfactual(model, x_in: torch.Tensor, steps: int = 80, lr: float = 3e-2, tv: float = 1e-3):
    """Legacy wrapper."""
    return _steering_counterfactual_robust(model, x_in, steps=steps, lr=lr, tv=tv)

def _yolop_counterfactual_robust(model, x_in: torch.Tensor, selector, steps: int = 80,
                                  lr: float = 2e-2, tv: float = 1e-3,
                                  target_flip_threshold: float = 0.1) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    ROBUST counterfactual for YOLOP model - ALWAYS returns a result.
    
    OPTIMIZED: Eliminated duplicate model forward passes in tracking loop.
    Now reuses forward pass result from loss computation for tracking.
    
    Returns the perturbation that causes the maximum probability change.
    If threshold is achieved, returns minimal perturbation among successes.
    
    Paper Algorithm 1: K=80 CF steps for decisive map generation.
    """
    device = x_in.device
    x_cf = x_in.detach().clone().requires_grad_(True)
    
    with torch.no_grad():
        out_orig = model(x_in)
        head_orig = selector(out_orig)
        prob_orig = _yolop_prob_mean(head_orig).item()
    
    # Target: flip probability direction
    if prob_orig > 0.5:
        target_prob = max(0.0, prob_orig - target_flip_threshold)
    else:
        target_prob = min(1.0, prob_orig + target_flip_threshold)
    
    target_tensor = torch.tensor(target_prob, device=device, dtype=x_in.dtype)
    
    opt = torch.optim.Adam([x_cf], lr=lr)
    
    # Track TWO things
    best_success_cf = None
    best_success_diff = float('inf')
    best_effort_cf = x_cf.detach().clone()
    best_effort_change = 0.0
    
    for step in range(steps):
        opt.zero_grad()
        out = model(x_cf)
        head = selector(out)
        prob_mean = _yolop_prob_mean(head)
        
        # OPTIMIZATION: Track BEFORE opt.step() to reuse forward pass result
        # This correctly associates output with the current x_cf state
        with torch.no_grad():
            prob_current = prob_mean.detach().item()
            current_change = abs(prob_current - prob_orig)
            current_diff = (x_cf - x_in).abs().mean().item()
            
            # Track best effort
            if current_change > best_effort_change:
                best_effort_change = current_change
                best_effort_cf = x_cf.detach().clone()
            
            # Track best success
            if current_change >= target_flip_threshold * 0.8:
                if current_diff < best_success_diff:
                    best_success_diff = current_diff
                    best_success_cf = x_cf.detach().clone()
        
        # Loss: approach target prob + minimize perturbation + smoothness
        output_loss = F.mse_loss(prob_mean, target_tensor)
        perturbation_loss = (x_cf - x_in).abs().mean()
        tv_loss_val = _tv_loss(x_cf, tv)
        
        # Adaptive weighting
        output_weight = 1.0
        perturb_weight = 0.1 if step < steps // 2 else 0.3
        
        loss = output_weight * output_loss + perturb_weight * perturbation_loss + tv_loss_val
        loss.backward()
        opt.step()
        
        with torch.no_grad():
            x_cf.data = _clamp01(x_cf.data)
    
    # Return best success if available, otherwise best effort
    if best_success_cf is not None:
        result_cf = best_success_cf
    else:
        result_cf = best_effort_cf
    
    with torch.no_grad():
        diff = (result_cf - x_in).abs().mean(dim=1, keepdim=True)
    
    return result_cf, diff

# Keep old names for compatibility
def _yolop_counterfactual_minimal(model, x_in: torch.Tensor, selector, steps: int = 80,
                                   lr: float = 2e-2, tv: float = 1e-3,
                                   target_flip_threshold: float = 0.1) -> Tuple[torch.Tensor, torch.Tensor]:
    """Wrapper that calls robust version."""
    return _yolop_counterfactual_robust(model, x_in, selector, steps=steps, lr=lr, tv=tv,
                                        target_flip_threshold=target_flip_threshold)

def _yolop_counterfactual(model, x_in: torch.Tensor, selector, steps: int = 80, lr: float = 2e-2, tv: float = 1e-3):
    """Legacy wrapper."""
    return _yolop_counterfactual_robust(model, x_in, selector, steps=steps, lr=lr, tv=tv)

def compute_dff_loss_cf_steering(model, preprocess, img_real: Image.Image, img_gen: Image.Image, 
                                  *, steps: int = 80, lr: float = 3e-2, tv: float = 1e-3, 
                                  downsample: int = 16) -> Optional[float]:
    """
    Compute DFF counterfactual LOSS for steering model (LOWER is better).
    MSE between counterfactual change maps on pooled 16×16 decisive maps.
    Paper Section IV-B: "We use MSE as D, computed on these averaged, pooled 16×16 decisive maps."
    
    FIXED: Now ALWAYS returns a float (never None unless actual exception).
    """
    if model is None or preprocess is None:
        return None
    try:
        xr = preprocess(img_real)
        xg = preprocess(img_gen)
        xr = F.interpolate(xr, size=(66, 200), mode="bilinear", align_corners=False)
        xg = F.interpolate(xg, size=(66, 200), mode="bilinear", align_corners=False)
        
        _, diff_r = _steering_counterfactual_robust(model, xr, steps=steps, lr=lr, tv=tv)
        _, diff_g = _steering_counterfactual_robust(model, xg, steps=steps, lr=lr, tv=tv)
        
        ds = max(4, int(downsample))  # Paper: pooled 16×16
        diff_r_ds = F.adaptive_avg_pool2d(diff_r, (ds, ds))
        diff_g_ds = F.adaptive_avg_pool2d(diff_g, (ds, ds))
        return float(F.mse_loss(diff_r_ds, diff_g_ds).item())
    except Exception as e:
        if not QUIET:
            print(f"[compute_dff_loss_cf_steering] Exception: {e}")
        return None
# Legacy alias (DEPRECATED - use compute_dff_loss_cf_steering instead)
# Will be removed in future versions
compute_rff_loss_cf_steering = compute_dff_loss_cf_steering

def compute_dff_loss_cf_yolop(model, preprocess, img_real: Image.Image, img_gen: Image.Image, 
                                  *, head: str = "da", steps: int = 80, lr: float = 2e-2, 
                                  tv: float = 1e-3, downsample: int = 16) -> Optional[float]:
    """
    Compute DFF counterfactual LOSS for YOLOP model (LOWER is better).
    MSE between counterfactual change maps on pooled 16×16 decisive maps.
    Paper Section IV-B: "We use MSE as D, computed on these averaged, pooled 16×16 decisive maps."
    
    FIXED: Now ALWAYS returns a float (never None unless actual exception).
    """
    if model is None or preprocess is None:
        return None
    try:
        xr = preprocess(img_real)
        xg = preprocess(img_gen)
        xr = F.interpolate(xr, size=(384, 640), mode="bilinear", align_corners=False)
        xg = F.interpolate(xg, size=(384, 640), mode="bilinear", align_corners=False)
        selector = yolop_select_da_head if head == "da" else yolop_select_ll_head
        
        _, diff_r = _yolop_counterfactual_robust(model, xr, selector, steps=steps, lr=lr, tv=tv)
        _, diff_g = _yolop_counterfactual_robust(model, xg, selector, steps=steps, lr=lr, tv=tv)
        
        ds = max(4, int(downsample))  # Paper: pooled 16×16
        diff_r_ds = F.adaptive_avg_pool2d(diff_r, (ds, ds))
        diff_g_ds = F.adaptive_avg_pool2d(diff_g, (ds, ds))
        return float(F.mse_loss(diff_r_ds, diff_g_ds).item())
    except Exception as e:
        if not QUIET:
            print(f"[compute_dff_loss_cf_yolop] Exception: {e}")
        return None
# Legacy alias (DEPRECATED - use compute_dff_loss_cf_yolop instead)
# Will be removed in future versions
compute_rff_loss_cf_yolop = compute_dff_loss_cf_yolop

print("✅ Cell 3: Core metrics loaded with ROBUST counterfactual implementation + STABILITY FIXES")
print("   - Counterfactual functions now ALWAYS return a result (no more None)")
print("   - Uses Adam optimizer with adaptive perturbation penalty")
print("   - Falls back to 'best effort' if threshold not achieved")
print("   OPTIMIZATIONS APPLIED:")
print("   - Cached base SDXL pipeline (5-10x faster model switching)")
print("   - torch.compile DISABLED by default (prevents SigmoidBackward0 errors)")  
print("   - Eliminated duplicate model forward passes in CF loops (1.3-1.5x CF speedup)")
print("   - TF32 warnings suppressed and precision settings applied")


✅ Cell 3: Core metrics loaded with ROBUST counterfactual implementation + STABILITY FIXES
   - Counterfactual functions now ALWAYS return a result (no more None)
   - Uses Adam optimizer with adaptive perturbation penalty
   - Falls back to 'best effort' if threshold not achieved
   OPTIMIZATIONS APPLIED:
   - Cached base SDXL pipeline (5-10x faster model switching)
   - torch.compile DISABLED by default (prevents SigmoidBackward0 errors)
   - Eliminated duplicate model forward passes in CF loops (1.3-1.5x CF speedup)
   - TF32 warnings suppressed and precision settings applied


In [4]:
# =============================================================================
# Shared sampling and probability utilities for RQ1/RQ2
# =============================================================================
# TERMINOLOGY: DFF = Decisive Feature Fidelity (paper term, was RFF in legacy code)
#
# === PAPER ALIGNMENT: Key Hyperparameters ===
# Paper Algorithm 1 (DFF-Guided Calibration Training):
#   - T = 22000 training steps
#   - β = 0.3 (OV loss weight)
#   - λ_dff = 0.08 (DFF loss weight)
#   - K = 80 CF steps for decisive map generation
#   - Pooled decisive maps: 16×16 resolution
#
# Paper Section V-A (Experimental Setup):
#   - N = 2126 matched KITTI–VirtualKITTI2 pairs
#   - 5 VKITTI2 scenes: Scene01, 02, 06, 18, 20
#   - Split: 80% calibration (Scene01, 02, 06) / 20% held-out (Scene18, 20)
#   - 1,701 calibration + 425 held-out items

import os, random, math, gc, contextlib, sys, shutil
from typing import Optional
from PIL import Image
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# Global quiet mode to suppress progress bars and intermediate prints
QUIET = False  # Enabled for detailed progress tracking during critical evaluation

# Reduce external library noise
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

# =============================================================================
# PAPER MODE FLAGS (align notebook outputs with paper)
# =============================================================================
PAPER_MODE = True
PAPER_OVF_STRICT_IOU = True       # Enforce IoU for LL to match paper definition
PAPER_DFF_USE_CF = True           # Use CF-XAI pooled MSE for DFF distance
PAPER_DFF_K = 80                  # Number of CF seeds for averaging
PAPER_DFF_DOWNSAMPLE = 16         # Pool to 16×16 as in paper

@contextlib.contextmanager
def suppress_output(enabled: bool = None):
    """Context manager to silence stdout/stderr when enabled is True."""
    if enabled is None:
        enabled = bool(globals().get('QUIET', False))
    if not enabled:
        yield
        return
    try:
        devnull = open(os.devnull, 'w')
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield
    finally:
        try:
            devnull.close()
        except Exception:
            pass

def cleanup_gpu_memory():
    """Clean up GPU memory after heavy operations."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def collect_shared_samples(n: int, split: str = 'heldout', allow_replacement: bool = False):
    """
    Collect n unique sample pairs from the specified split.
    
    Per Paper Section V-A:
    - 'calibration': Scenes 01, 02, 06 (80%, ~1701 items) - for threshold determination
    - 'heldout': Scenes 18, 20 (20%, ~425 items) - for final evaluation
    
    Args:
        n: Number of samples to collect
        split: 'calibration' or 'heldout'
        allow_replacement: If False (default), samples are unique; if True, allows duplicates
    
    Returns:
        List of (kitti_path, seg_path, seq_id, frame_num) tuples
    """
    pairs = []
    seen = set()
    max_attempts = n * 10  # Prevent infinite loop if not enough unique samples
    attempts = 0
    
    while len(pairs) < n and attempts < max_attempts:
        attempts += 1
        kitti_path, seg_path, seq_id, frame_num = find_matching_kitti_seg_pair(split=split)
        key = (seq_id, frame_num)
        
        if allow_replacement or key not in seen:
            pairs.append((kitti_path, seg_path, seq_id, frame_num))
            seen.add(key)
    
    if len(pairs) < n:
        print(f"⚠️ WARNING: Only found {len(pairs)} unique samples (requested {n})")
    
    return pairs

def wilson_ci(k: int, n: int, z: float = 1.96):
    """
    Wilson score interval for binomial proportion.
    Better coverage than normal approximation, especially for small n.
    
    Per Paper: Used for pass-rate confidence intervals.
    
    Args:
        k: Number of successes
        n: Number of trials
        z: Z-score for confidence level (1.96 for 95% CI)
    
    Returns:
        (center, lower_bound, upper_bound) tuple
    """
    if n == 0:
        return 0.0, 0.0, 0.0
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (z*math.sqrt((p*(1-p) + z*z/(4*n))/n)) / denom
    return center, max(0.0, center - half), min(1.0, center + half)

# =============================================================================
# PAPER-ALIGNED CONFIGURATION
# =============================================================================
# Per Paper Section V-A: N = 2126 matched pairs total
# - For PRODUCTION/PAPER results: Set N_SAMPLES = 2126 (or use full dataset)
# - For QUICK TESTING: N_SAMPLES = 30-100 is acceptable for development
# - MINIMUM for statistical validity: N_SAMPLES ≥ 30 (Wilson CI requirement)
N_SAMPLES = 100  # Dataset-wise experiment sample size

RESULTS_DIR = os.path.join(os.getcwd(), 'results')
# Clean up previous results as requested
if os.path.exists(RESULTS_DIR):
    print(f"Cleaning up previous results in {RESULTS_DIR}...")
    try:
        shutil.rmtree(RESULTS_DIR)
    except Exception as e:
        print(f"Warning: Could not delete {RESULTS_DIR}: {e}")
os.makedirs(RESULTS_DIR, exist_ok=True)

TARGET_SIZE = (1248, 376)  # Native KITTI resolution (Paper Section V-A)
SEED = 12345  # Fixed seed for reproducibility

# =============================================================================
# THRESHOLD CONFIGURATION - PAPER ALIGNMENT
# =============================================================================
# Per Paper Section V-C: δ = -0.005 for steering, δ = -0.010 for segmentation
# These represent the boundary for Non-Inferiority (NI)
#
# === METRIC SEMANTICS (CRITICAL FOR CORRECT INTERPRETATION) ===
#
# OVF (Output-Value Fidelity): HIGHER is better (similarity score in [0,1])
#   - Paper Table V reports ΔOV: positive = improvement
#   - Paper: "For OV (similarity, higher is better): positive ΔOV means improvement"
#   - Success: OVF >= TAU_OVF
#
# DFF_distance (Decisive Feature Fidelity): LOWER is better (distance in [0,1])
#   - Paper Table V reports ΔDFF: negative = improvement
#   - Paper: "For DFF (distance, lower is better): negative ΔDFF means improvement"
#   - Success: DFF_distance <= TAU_DFF_DISTANCE
#   - DFF_distance computed via CF-XAI pooled MSE (paper definition)
#
# IV (Input-Value Fidelity): HIGHER is better (1 - LPIPS, in [0,1])
#   - Paper Table V reports ΔIV: positive = improvement
#
# LEGACY NAMING: RFF was the old code name - DFF is the paper-correct term
# =============================================================================

TAU_OVF = 0.7               # Output fidelity threshold (HIGHER is better)
TAU_DFF = 0.7               # DEPRECATED: Legacy similarity threshold - DO NOT USE
TAU_DFF_DISTANCE = 0.3      # Default DFF DISTANCE threshold (LOWER is better)

# =============================================================================
# PER-SUT DFF THRESHOLDS - Paper Table II
# =============================================================================
# Calibrated from calibration split as 90th and 95th percentiles.
# A held-out synthetic image passes if its DFF distance is below epsilon.
# 
# Paper Table II:
# | SUT       | ε₉₀      | ε₉₅        |
# |-----------|----------|------------|
# | Steering  | 0.064245 | 0.08738442 |
# | YOLOP–DA  | 0.570196 | 0.6990833  |
# | YOLOP–LL  | 0.582134 | 0.7362408  |

TAU_DFF_DISTANCE_PER_SUT = {
    'steering': {
        'eps_90': 0.064245,      # ε₉₀ for steering (Paper Table II)
        'eps_95': 0.08738442,    # ε₉₅ for steering (Paper Table II: \epsSteer)
    },
    'da': {
        'eps_90': 0.570196,      # ε₉₀ for YOLOP-DA (Paper Table II: \epsDANinety)
        'eps_95': 0.6990833,     # ε₉₅ for YOLOP-DA (Paper Table II: \epsDA)
    },
    'll': {
        'eps_90': 0.582134,      # ε₉₀ for YOLOP-LL (Paper Table II: \epsLLNinety)
        'eps_95': 0.7362408,     # ε₉₅ for YOLOP-LL (Paper Table II: \epsLL)
    },
}

def get_dff_threshold(sut: str, percentile: int = 95) -> float:
    """Get the DFF distance threshold for a specific SUT (per Paper Table II).
    
    Args:
        sut: One of 'steering', 'da', 'll'
        percentile: Either 90 or 95 (default 95)
    
    Returns:
        The calibrated DFF distance threshold for that SUT
    """
    key = f'eps_{percentile}'
    return TAU_DFF_DISTANCE_PER_SUT.get(sut, {}).get(key, TAU_DFF_DISTANCE)

# Legacy aliases for backward compatibility (DEPRECATED - will be removed in future versions)
# Use TAU_DFF and TAU_DFF_DISTANCE instead
TAU_RFF = TAU_DFF
TAU_RFF_DISTANCE = TAU_DFF_DISTANCE

# Combined success predicate (per paper):
#   SUCCESS = (OVF >= TAU_OVF) AND (DFF_distance <= TAU_DFF_DISTANCE)

# =============================================================================
# GPU Configuration with Reproducibility Settings
# =============================================================================
# OPTIMIZATION: Deterministic mode disabled for 10-30% speedup
# Set ENABLE_DETERMINISM = True for exact reproducibility (slower)
# =============================================================================
ENABLE_DETERMINISM = False  # False = faster (10-30%), True = exact reproducibility
FAST_MODE = False  # EXPERIMENTAL: Set True for reduced CF steps (80->40) in exploratory runs

USE_GPU = torch.cuda.is_available()
if USE_GPU:
    GPU_DEVICE = torch.cuda.get_device_name(0)
    TORCH_DTYPE = torch.float16  # Use FP16 for faster inference on modern GPUs
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True  # Enable cuDNN auto-tuner
    
    if ENABLE_DETERMINISM:
        # CRITICAL: Enable deterministic operations for reproducibility (slower)
        torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:
            pass  # Older PyTorch versions may not support this
        if not QUIET:
            print(f"✓ GPU Enabled: {GPU_DEVICE}")
            print(f"  - Using FP16 precision for faster inference")
            print(f"  - cuDNN benchmark mode enabled")
            print(f"  - Deterministic mode enabled for reproducibility (slower)")
    else:
        # OPTIMIZATION: Non-deterministic mode for 10-30% speedup
        torch.backends.cudnn.deterministic = False
        if not QUIET:
            print(f"✓ GPU Enabled: {GPU_DEVICE}")
            print(f"  - Using FP16 precision for faster inference")
            print(f"  - cuDNN benchmark mode enabled")
            print(f"  - Deterministic mode DISABLED for speed (set ENABLE_DETERMINISM=True for reproducibility)")
    
    if FAST_MODE and not QUIET:
        print(f"  - FAST_MODE enabled: CF steps reduced (80→40) for exploratory runs")
else:
    TORCH_DTYPE = torch.float32
    GPU_DEVICE = "CPU"
    if not QUIET:
        print("GPU not available, using CPU with FP32 precision")
    # Auto-diagnose and fix CUDA issues
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        if result.returncode == 0:
            print("\n⚠️  nvidia-smi found but torch.cuda.is_available() returned False")
            print("    This may be a PyTorch installation issue. Suggested fix:")
            print("    pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
        else:
            print("\n✗ NVIDIA GPU drivers not detected.")
            print("    Install NVIDIA drivers: https://www.nvidia.com/Download/driverDetails.aspx")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        print("\n✗ NVIDIA drivers not found (nvidia-smi not in PATH)")
        print("    Install NVIDIA drivers: https://www.nvidia.com/Download/driverDetails.aspx")
    except Exception:
        pass

# =============================================================================
# Configuration Summary (Paper Alignment)
# =============================================================================
if not QUIET:
    print("\n" + "="*70)
    print("CONFIGURATION (Paper Aligned)")
    print("="*70)
    print(f"N_SAMPLES = {N_SAMPLES} (Paper: 2126 matched pairs)")
    print(f"TARGET_SIZE = {TARGET_SIZE} (Native KITTI resolution)")
    print(f"GPU = {GPU_DEVICE if USE_GPU else 'CPU'}")
    print()
    print("THRESHOLDS:")
    print(f"  TAU_OVF = {TAU_OVF} (Output Fidelity, HIGHER is better)")
    print(f"  TAU_DFF_DISTANCE = {TAU_DFF_DISTANCE} (DFF Distance, LOWER is better)")
    print()
    print("PER-SUT DFF THRESHOLDS (Paper Table II):")
    for sut, thresholds in TAU_DFF_DISTANCE_PER_SUT.items():
        print(f"  {sut}: ε₉₀={thresholds['eps_90']:.6f}, ε₉₅={thresholds['eps_95']:.6f}")
    print()
    print("SUCCESS PREDICATE: (OVF >= TAU_OVF) AND (DFF_distance <= ε)")
    print()
    
    if N_SAMPLES < 30:
        print(f"⚠️  WARNING: N_SAMPLES={N_SAMPLES} < 30. Results lack statistical validity!")
        print(f"   Wilson CI requires n≥30 for reliable inference.")
    else:
        print(f"✓ Sample size N={N_SAMPLES} is sufficient for statistical inference")

# =============================================================================
# PAPER-ALIGNED OVF AND DFF OVERRIDES
# =============================================================================
# Ensure LL uses IoU (not tolerant metrics) for OVF per paper definition
# and DFF distance uses CF-XAI pooled MSE with K-seed averaging.
try:
    _compute_ovf_yolop_similarity_base = compute_ovf_yolop_similarity
except NameError:
    _compute_ovf_yolop_similarity_base = None

def compute_ovf_yolop_similarity(model, preprocess, img_real: Image.Image, img_gen: Image.Image,
                                  head: str = "da", use_tolerant_metric: bool = None) -> Optional[float]:
    if use_tolerant_metric is None:
        use_tolerant_metric = not PAPER_OVF_STRICT_IOU
    if PAPER_OVF_STRICT_IOU and head == "ll":
        use_tolerant_metric = False
    if _compute_ovf_yolop_similarity_base is None:
        return None
    return _compute_ovf_yolop_similarity_base(model, preprocess, img_real, img_gen, head=head, use_tolerant_metric=use_tolerant_metric)

# Update alias to use strict wrapper
compute_ovf_yolop = compute_ovf_yolop_similarity

try:
    _compute_dff_distance_saliency = compute_dff_distance
except NameError:
    _compute_dff_distance_saliency = None

def _avg_cf_map_steering(model, x_in: torch.Tensor, *, k_cf: int, downsample: int, seed_base: int = 0):
    maps = []
    for k in range(k_cf):
        torch.manual_seed(seed_base + k)
        _, diff = _steering_counterfactual_robust(model, x_in, steps=PAPER_DFF_K, lr=3e-2, tv=1e-3)
        diff_ds = F.adaptive_avg_pool2d(diff, (downsample, downsample))
        maps.append(diff_ds)
    return torch.stack(maps, dim=0).mean(dim=0)

def _avg_cf_map_yolop(model, x_in: torch.Tensor, selector, *, k_cf: int, downsample: int, seed_base: int = 0):
    maps = []
    for k in range(k_cf):
        torch.manual_seed(seed_base + k)
        _, diff = _yolop_counterfactual_robust(model, x_in, selector, steps=PAPER_DFF_K, lr=2e-2, tv=1e-3)
        diff_ds = F.adaptive_avg_pool2d(diff, (downsample, downsample))
        maps.append(diff_ds)
    return torch.stack(maps, dim=0).mean(dim=0)

def compute_dff_distance_cf_steering(model, preprocess, img_real: Image.Image, img_gen: Image.Image,
                                     *, k_cf: int = None, downsample: int = None, seed_base: int = None) -> Optional[float]:
    if model is None or preprocess is None:
        return None
    try:
        k_cf = k_cf or PAPER_DFF_K
        downsample = downsample or PAPER_DFF_DOWNSAMPLE
        seed_base = seed_base if seed_base is not None else SEED
        xr = preprocess(img_real)
        xg = preprocess(img_gen)
        xr = F.interpolate(xr, size=(66, 200), mode="bilinear", align_corners=False)
        xg = F.interpolate(xg, size=(66, 200), mode="bilinear", align_corners=False)
        avg_r = _avg_cf_map_steering(model, xr, k_cf=k_cf, downsample=downsample, seed_base=seed_base)
        avg_g = _avg_cf_map_steering(model, xg, k_cf=k_cf, downsample=downsample, seed_base=seed_base + 1000)
        return float(F.mse_loss(avg_r, avg_g).item())
    except Exception as e:
        if not QUIET:
            print(f"[compute_dff_distance_cf_steering] Exception: {e}")
        return None

def compute_dff_distance_cf_yolop(model, preprocess, img_real: Image.Image, img_gen: Image.Image, selector,
                                   *, k_cf: int = None, downsample: int = None, seed_base: int = None) -> Optional[float]:
    if model is None or preprocess is None:
        return None
    try:
        k_cf = k_cf or PAPER_DFF_K
        downsample = downsample or PAPER_DFF_DOWNSAMPLE
        seed_base = seed_base if seed_base is not None else SEED
        xr = preprocess(img_real)
        xg = preprocess(img_gen)
        xr = F.interpolate(xr, size=(384, 640), mode="bilinear", align_corners=False)
        xg = F.interpolate(xg, size=(384, 640), mode="bilinear", align_corners=False)
        avg_r = _avg_cf_map_yolop(model, xr, selector, k_cf=k_cf, downsample=downsample, seed_base=seed_base)
        avg_g = _avg_cf_map_yolop(model, xg, selector, k_cf=k_cf, downsample=downsample, seed_base=seed_base + 1000)
        return float(F.mse_loss(avg_r, avg_g).item())
    except Exception as e:
        if not QUIET:
            print(f"[compute_dff_distance_cf_yolop] Exception: {e}")
        return None

def compute_dff_distance(model, preprocess, img_real: Image.Image, img_gen: Image.Image, output_selector=None,
                         *, use_cf: bool = None, k_cf: int = None, downsample: int = None) -> Optional[float]:
    if use_cf is None:
        use_cf = PAPER_DFF_USE_CF
    if use_cf:
        if output_selector is None:
            return compute_dff_distance_cf_steering(model, preprocess, img_real, img_gen, k_cf=k_cf, downsample=downsample)
        return compute_dff_distance_cf_yolop(model, preprocess, img_real, img_gen, output_selector, k_cf=k_cf, downsample=downsample)
    if _compute_dff_distance_saliency is None:
        return None
    return _compute_dff_distance_saliency(model, preprocess, img_real, img_gen, output_selector=output_selector)

# Legacy alias for backward compatibility
compute_rff_distance = compute_dff_distance

Cleaning up previous results in /networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/results...
✓ GPU Enabled: NVIDIA L40
  - Using FP16 precision for faster inference
  - cuDNN benchmark mode enabled
  - Deterministic mode DISABLED for speed (set ENABLE_DETERMINISM=True for reproducibility)

CONFIGURATION (Paper Aligned)
N_SAMPLES = 100 (Paper: 2126 matched pairs)
TARGET_SIZE = (1248, 376) (Native KITTI resolution)
GPU = NVIDIA L40

THRESHOLDS:
  TAU_OVF = 0.7 (Output Fidelity, HIGHER is better)
  TAU_DFF_DISTANCE = 0.3 (DFF Distance, LOWER is better)

PER-SUT DFF THRESHOLDS (Paper Table II):
  steering: ε₉₀=0.064245, ε₉₅=0.087384
  da: ε₉₀=0.570196, ε₉₅=0.699083
  ll: ε₉₀=0.582134, ε₉₅=0.736241

SUCCESS PREDICATE: (OVF >= TAU_OVF) AND (DFF_distance <= ε)

✓ Sample size N=100 is sufficient for statistical inference


In [5]:
# =============================================================================
# Single matched-pair diagnostics - CORRESPONDS TO PAPER TABLE IV
# =============================================================================
# TERMINOLOGY: DFF = Decisive Feature Fidelity (paper term, was RFF in legacy code)
#
# === PAPER TABLE IV ALIGNMENT ===
# Table IV shows pilot test results for one matched pair (Scene20, frame 750):
#   - LPIPS: Perceptual distance (lower = better)
#   - OV loss: Raw task loss (MSE for steering, 1-IoU for YOLOP)
#   - DFF: Decisive-feature distance (lower = better)
# 
# Variants:
#   - Baseline: Uncalibrated VirtualKITTI2
#   - OVF-X: Calibration optimizing only OV loss for SUT X
#   - DFF-X: Calibration using full objective (Eq. 5: L_recon + β·L_OV + λ_dff·L_DFF)

import pandas as pd
import numpy as np
import gc

if 'generate_vkitti_image' not in globals():
    def generate_vkitti_image(pipe, conditioning: Image.Image, *, seed: int, steps: int = 80, guidance: float = 7.5, cond_scale: float = 0.8) -> Image.Image:
        """Fallback SDXL ControlNet wrapper to mirror training-time prompts."""
        prompt = "Transform real KITTI driving scene to V-KITTI virtual style"
        negative_prompt = (
            "blurry, low quality, distorted, cartoon, painting, sketch, artifacts, noise, real photography, KITTI style"
        )
        generator = torch.Generator(device="cuda" if USE_GPU else "cpu").manual_seed(seed)
        with suppress_output(globals().get('QUIET', False)):
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=conditioning,
                num_inference_steps=steps,
                guidance_scale=guidance,
                controlnet_conditioning_scale=cond_scale,
                generator=generator
            )
        return result.images[0]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Fixed pilot pair to match paper (Scene20, frame 750)
_pilot_seq = "0020"
_pilot_scene = "Scene20"
_pilot_frame = 750
kitti_path = os.path.join(KITTI_PATH, _pilot_seq, f"{_pilot_frame:06d}.png")
seg_path = os.path.join(VKITTI_SEG_PATH, _pilot_scene, f"classgt_{_pilot_frame:05d}.png")

if not (os.path.exists(kitti_path) and os.path.exists(seg_path)):
    print("⚠️ Fixed pilot pair not found. Falling back to a random held-out sample.")
    sample = collect_shared_samples(1)[0]
    kitti_path, seg_path, seq_id, frame_num = sample
else:
    seq_id, frame_num = _pilot_seq, _pilot_frame

conditioning, kitti_img, _, _, _ = prepare_proper_conditioning(kitti_path, seg_path, target_size=TARGET_SIZE)

yolop_model, yolop_pre = load_yolop_sut()
steering_model, steering_pre = load_steering_sut()

def _lpips_distance(img_real: Image.Image, img_gen: Image.Image) -> float:
    """Compute LPIPS perceptual distance between two images."""
    model = _get_lpips()
    if model is None:
        return float("nan")
    arr_real = np.array(img_real).astype(np.float32) / 255.0
    arr_gen = np.array(img_gen).astype(np.float32) / 255.0
    t_real = torch.from_numpy(arr_real).permute(2, 0, 1).unsqueeze(0)
    t_gen = torch.from_numpy(arr_gen).permute(2, 0, 1).unsqueeze(0)
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device('cuda' if USE_GPU else 'cpu')
    t_real = t_real.to(device)
    t_gen = t_gen.to(device)
    with torch.no_grad():
        dist = model(t_real, t_gen).item()
    return float(dist)

# Generator variants: DFF-* = Decisive Feature Fidelity trained (paper terminology)
# Maps to Paper Table IV columns: Baseline, OVF-X, DFF-X
sut_variants = {
    "steering": [
        ("Baseline", "Baseline"),      # Paper: "Baseline" (uncalibrated)
        ("OVF-Steer", "OVF-Steering"),  # Paper: "OVF-Steer" (OV-only calibration)
        ("DFF-Steer", "DFF-Steering-CF"),  # Paper: "DFF-Steer" (full objective)
    ],
    "da": [
        ("Baseline", "Baseline"),
        ("OVF-DA", "OVF-YOLOP-DA"),
        ("DFF-DA", "DFF-YOLOP-DA-CF"),
    ],
    "ll": [
        ("Baseline", "Baseline"),
        ("OVF-LL", "OVF-YOLOP-LL"),
        ("DFF-LL", "DFF-YOLOP-LL-CF"),
    ],
}

def _dff_selector_da(out):
    return _yolop_prob_mean(yolop_select_da_head(out))

def _dff_selector_ll(out):
    return _yolop_prob_mean(yolop_select_ll_head(out))

# Legacy aliases for backward compatibility
_rff_selector_da = _dff_selector_da
_rff_selector_ll = _dff_selector_ll

rows = []

print("="*70)
print("PAPER TABLE IV: Pilot test on one matched pair")
print("="*70)
print(f"Selected sample:")
print(f"  Sequence: {seq_id}")
print(f"  Frame:    {frame_num}")
print(f"  KITTI:    {kitti_path}")
print(f"  SEG:      {seg_path}")

for sut, variants in sut_variants.items():
    for label, gen_name in variants:
        model_path = GENERATORS.get(gen_name)
        if not model_path:
            rows.append({
                "SUT": sut,
                "Variant": label,
                "LPIPS": np.nan,      # Paper column name
                "OV_loss": np.nan,    # Paper column name
                "DFF": np.nan,        # Paper column name
                "note": "missing checkpoint",
            })
            continue
        try:
            pipe = load_trained_pipeline(model_path)
            gen_img = generate_vkitti_image(pipe, conditioning, seed=SEED)
        except Exception as exc:
            rows.append({
                "SUT": sut,
                "Variant": label,
                "LPIPS": np.nan,
                "OV_loss": np.nan,
                "DFF": np.nan,
                "note": f"generation failed: {exc}"
            })
            continue
        finally:
            try:
                del pipe
            except UnboundLocalError:
                pass
            gc.collect()
            if USE_GPU:
                torch.cuda.empty_cache()

        lpips_dist = _lpips_distance(kitti_img, gen_img)

        ovf_loss = np.nan
        dff_dist = np.nan  # DFF as distance (lower = better) - paper terminology

        if sut == "steering" and steering_model and steering_pre:
            ovf_loss = compute_ovf_loss_steering(steering_model, steering_pre, kitti_img, gen_img)
            dff_dist = compute_dff_distance(steering_model, steering_pre, kitti_img, gen_img)
        elif sut == "da" and yolop_model and yolop_pre:
            ovf_loss = compute_ovf_loss_yolop(yolop_model, yolop_pre, kitti_img, gen_img, head="da")
            dff_dist = compute_dff_distance(yolop_model, yolop_pre, kitti_img, gen_img, output_selector=_dff_selector_da)
        elif sut == "ll" and yolop_model and yolop_pre:
            ovf_loss = compute_ovf_loss_yolop(yolop_model, yolop_pre, kitti_img, gen_img, head="ll")
            dff_dist = compute_dff_distance(yolop_model, yolop_pre, kitti_img, gen_img, output_selector=_dff_selector_ll)

        rows.append({
            "SUT": sut,
            "Variant": label,
            "LPIPS": lpips_dist,   # Paper column name
            "OV_loss": ovf_loss,   # Paper column name  
            "DFF": dff_dist,       # Paper column name (was RFF in legacy code)
        })

single_sample_df = pd.DataFrame(rows)
single_sample_df["SUT"] = pd.Categorical(single_sample_df["SUT"], categories=["steering", "da", "ll"], ordered=True)
single_sample_df["Variant"] = pd.Categorical(
    single_sample_df["Variant"], 
    categories=["Baseline", "OVF-Steer", "DFF-Steer", "OVF-DA", "DFF-DA", "OVF-LL", "DFF-LL"], 
    ordered=True
)
single_sample_df.sort_values(["SUT", "Variant"], inplace=True)
single_sample_df.reset_index(drop=True, inplace=True)

# Format for display matching Paper Table IV
display_cols = ["SUT", "Variant", "LPIPS", "OV_loss", "DFF"]
formatted = single_sample_df[display_cols].copy()
for col in ["LPIPS", "OV_loss", "DFF"]:
    formatted[col] = formatted[col].apply(
        lambda x: f"{x:.4f}" if isinstance(x, (float, np.floating)) and not np.isnan(x) else "nan"
    )

print("\n" + "="*70)
print("TABLE IV FORMAT (Lower is better for all metrics)")
print("="*70)
print("Metric interpretation (per Paper Table IV):")
print("  LPIPS:   Perceptual distance (LOWER = more similar)")
print("  OV_loss: Output value loss (MSE for steering, 1-IoU for YOLOP)")
print("           LOWER = better output fidelity")
print("  DFF:     Decisive Feature Fidelity distance (LOWER = better)")
print("           DFF = paper terminology; 'RFF' was legacy code name")
print()
print(formatted.to_string(index=False))

# Paper Table IV reference values (Scene20, frame 750):
print("\n" + "-"*70)
print("PAPER TABLE IV REFERENCE VALUES (for comparison):")
print("-"*70)
paper_reference = """
| SUT   | Variant    | LPIPS | OV loss | DFF   |
|-------|------------|-------|---------|-------|
| Steer | Baseline   | 0.610 | 0.0006  | 0.361 |
| Steer | OVF-Steer  | 0.473 | 0.3908  | 0.278 |
| Steer | DFF-Steer  | 0.385 | 0.0028  | 0.229 |
| DA    | Baseline   | 0.610 | 0.0172  | 0.541 |
| DA    | OVF-DA     | 0.477 | 0.0044  | 0.630 |
| DA    | DFF-DA     | 0.390 | 0.0044  | 0.577 |
| LL    | Baseline   | 0.610 | 0.0019  | 0.712 |
| LL    | OVF-LL     | 0.421 | 0.0011  | 0.630 |
| LL    | DFF-LL     | 0.451 | 0.0013  | 0.635 |
"""
print(paper_reference)

SINGLE_SAMPLE_RESULTS = single_sample_df

/tmp/ipykernel_786027/1659505533.py:315: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(YOLOP_WEIGHTS_PATH, map_location="cpu")


  ℹ YOLOP torch.compile disabled for stability (prevents SigmoidBackward0 errors)


/tmp/ipykernel_786027/1659505533.py:383: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(weights_path, map_location="cpu")


  ℹ Steering torch.compile disabled for stability
PAPER TABLE IV: Pilot test on one matched pair
Selected sample:
  Sequence: 0020
  Frame:    750
  KITTI:    /networkhome/WMGDS/safaei_d/3_generators/data/data_for_new_mvp/KITTI/0020/000750.png
  SEG:      /networkhome/WMGDS/safaei_d/3_generators/data/data_for_new_mvp/class_segmentation_v_kitti/Scene20/classgt_00750.png


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/networkhome/WMGDS/safaei_d/miniconda3/envs/img2img/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/networkhome/WMGDS/safaei_d/miniconda3/envs/img2img/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /networkhome/WMGDS/safaei_d/miniconda3/envs/img2img/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth


/networkhome/WMGDS/safaei_d/miniconda3/envs/img2img/lib/python3.11/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch


TABLE IV FORMAT (Lower is better for all metrics)
Metric interpretation (per Paper Table IV):
  LPIPS:   Perceptual distance (LOWER = more similar)
  OV_loss: Output value loss (MSE for steering, 1-IoU for YOLOP)
           LOWER = better output fidelity
  DFF:     Decisive Feature Fidelity distance (LOWER = better)
           DFF = paper terminology; 'RFF' was legacy code name

     SUT   Variant  LPIPS OV_loss    DFF
steering  Baseline 0.5816  0.0168 0.0004
steering OVF-Steer 0.5960  0.2135 0.0005
steering DFF-Steer 0.6721  0.0638 0.0004
      da  Baseline 0.5816  0.0165 0.0006
      da    OVF-DA 0.4423  0.0028 0.0002
      da    DFF-DA 0.3336  0.0037 0.0002
      ll  Baseline 0.5816  0.0015 0.0000
      ll    OVF-LL 0.2502  0.0004 0.0000
      ll    DFF-LL 0.3049  0.0009 0.0000

----------------------------------------------------------------------
PAPER TABLE IV REFERENCE VALUES (for comparison):
----------------------------------------------------------------------

| SUT   | Var

In [6]:
# === Assessment: Determine experiment success based on evaluated numbers ===
# TERMINOLOGY: DFF = Decisive Feature Fidelity (paper term, was RFF in legacy code)
import math, json, datetime as dt
import numpy as np
import pandas as pd
from collections import defaultdict


def _wilson_fields(prob_df, sut, gen):
    """Extract Wilson CI fields from probability dataframe."""
    row = prob_df[(prob_df['SUT'] == sut) & (prob_df['generator'] == gen)]
    if len(row) == 0:
        return None
    r = row.iloc[0].to_dict()
    return {
        'n': int(r.get('n', 0) or 0), 
        'p_hat': float(r.get('p_hat', float('nan'))), 
        'ci_lo': float(r.get('ci_lo', float('nan'))), 
        'ci_hi': float(r.get('ci_hi', float('nan')))
    }


def _paired_keys(df, sut, gen_a, gen_b):
    """Get paired (seq, frame) keys that exist for both generators."""
    a = df[(df['SUT'] == sut) & (df['generator'] == gen_a)].dropna(subset=['seq', 'frame'])
    b = df[(df['SUT'] == sut) & (df['generator'] == gen_b)].dropna(subset=['seq', 'frame'])
    a_keys = set(zip(a['seq'], a['frame']))
    b_keys = set(zip(b['seq'], b['frame']))
    return sorted(a_keys & b_keys)


def _rows_by_key(df, sut, gen, key):
    """Get row dict for a specific (seq, frame) key."""
    seq, frame = key
    row = df[(df['SUT'] == sut) & (df['generator'] == gen) & (df['seq'] == seq) & (df['frame'] == frame)]
    return row.iloc[0].to_dict() if len(row) > 0 else None


def _binom_two_sided_p(wins, losses):
    """Two-sided binomial test p-value under H0: p=0.5."""
    n = wins + losses
    if n <= 0:
        return 1.0
    r = wins
    
    def _pmf(k):
        return math.comb(n, k) * (0.5 ** n)
    
    observed = _pmf(r)
    # Sum probabilities as or more extreme than observed (two-sided)
    p = sum(_pmf(k) for k in range(0, n + 1) if _pmf(k) <= observed + 1e-15)
    return min(1.0, p)


def _get_dff_col(df):
    """Get the DFF column name (with backward compatibility for legacy RFF naming)."""
    if 'DFF_distance' in df.columns:
        return 'DFF_distance'
    elif 'RFF_distance' in df.columns:  # Legacy compatibility
        return 'RFF_distance'
    elif 'DFF' in df.columns:
        return 'DFF'
    elif 'RFF' in df.columns:  # Legacy compatibility
        return 'RFF'
    else:
        raise ValueError("Neither 'DFF_distance' nor 'RFF_distance' column found in dataframe")

# Legacy alias for backward compatibility (DEPRECATED - use _get_dff_col instead)
_get_rff_col = _get_dff_col


def assess_rq1_success(df_rq1_multi, prob_rq1_multi, tau_ovf=0.7, tau_dff_dist=None, alpha=0.05, delta_prob=0.0):
    """
    Assess O1/RQ1 (Assessment): Does DFF reveal mechanism shifts that IV/OV miss?
    
    Paper Objective O1: Does DFF reveal mechanism shifts (high DFF distance) in synthetic 
    samples that otherwise pass standard IV and OV fidelity checks?
    
    Success criteria per SUT:
    1) Probability P(OVF>=tau_ovf & DFF_distance<=eps) for DFF generator > Baseline and > OVF,
       and Wilson lower bound for DFF exceeds comparator p_hat + delta_prob.
    2) Paired sign test on success predicate shows DFF beats comparator (p < alpha).
    
    CRITICAL PAPER ALIGNMENT:
    Uses per-SUT thresholds from Paper Table II (via get_dff_threshold) instead of a global threshold.
    
    Args:
        tau_ovf: OVF threshold (higher is better)
        tau_dff_dist: DEPRECATED/IGNORED. Uses per-SUT thresholds from Table II.
        alpha: Significance level for sign test
        delta_prob: Minimum probability improvement required
    
    Returns:
        Dict of booleans and details per SUT.
    """
    # Get DFF column name
    dff_col = _get_dff_col(df_rq1_multi)
    
    # Generator mapping: paper uses DFF, but checkpoint directories may use RFF or DFF
    # Support both legacy (RFF) and new (DFF) naming
    sut_sets = {
        'steering': {
            'dff': ['DFF-Steering-CF', 'RFF-Steering-CF'],  # Try DFF first, fall back to RFF
            'ovf': 'OVF-Steering', 
            'base': 'Baseline'
        },
        'da': {
            'dff': ['DFF-YOLOP-DA-CF', 'RFF-YOLOP-DA-CF'],
            'ovf': 'OVF-YOLOP-DA', 
            'base': 'Baseline'
        },
        'll': {
            'dff': ['DFF-YOLOP-LL-CF', 'RFF-YOLOP-LL-CF'],
            'ovf': 'OVF-YOLOP-LL', 
            'base': 'Baseline'
        },
    }
    
    def _find_gen(prob_df, sut, names):
        """Find first matching generator name from list."""
        if isinstance(names, str):
            return names
        for name in names:
            if len(prob_df[(prob_df['SUT'] == sut) & (prob_df['generator'] == name)]) > 0:
                return name
        return names[0]  # Default to first option
    
    results = {}
    
    for sut, gens in sut_sets.items():
        # CRITICAL: Use per-SUT threshold from Paper Table II
        # If get_dff_threshold is not available (e.g. cell not run), fall back to global
        try:
            sut_eps = get_dff_threshold(sut, 95)
        except NameError:
            sut_eps = globals().get('TAU_DFF_DISTANCE', 0.3)
            
        dff_gen = _find_gen(prob_rq1_multi, sut, gens['dff'])
        ovf = gens['ovf']
        base = gens['base']
        
        pf_dff = _wilson_fields(prob_rq1_multi, sut, dff_gen)
        pf_ovf = _wilson_fields(prob_rq1_multi, sut, ovf)
        pf_base = _wilson_fields(prob_rq1_multi, sut, base)
        
        comp = {}
        for label, pf, gen_name in [('ovf', pf_ovf, ovf), ('base', pf_base, base)]:
            ok_prob = False
            if pf_dff and pf:
                ok_prob = (pf_dff['p_hat'] > pf['p_hat'] + 1e-12) and (pf_dff['ci_lo'] > (pf['p_hat'] + delta_prob))
            
            # Paired success predicate comparison
            keys = _paired_keys(df_rq1_multi, sut, dff_gen, gen_name)
            wins = 0
            losses = 0
            
            for key in keys:
                ar = _rows_by_key(df_rq1_multi, sut, dff_gen, key)
                br = _rows_by_key(df_rq1_multi, sut, gen_name, key)
                if not ar or not br:
                    continue
                
                # Success predicate: OVF >= tau AND DFF_distance <= eps (per-SUT)
                # DFF_distance is LOWER is better
                a_ok = (
                    pd.notna(ar.get('OVF')) and 
                    pd.notna(ar.get(dff_col)) and 
                    (ar['OVF'] >= tau_ovf) and 
                    (ar[dff_col] <= sut_eps)
                )
                b_ok = (
                    pd.notna(br.get('OVF')) and 
                    pd.notna(br.get(dff_col)) and 
                    (br['OVF'] >= tau_ovf) and 
                    (br[dff_col] <= sut_eps)
                )
                
                if a_ok and not b_ok:
                    wins += 1
                elif b_ok and not a_ok:
                    losses += 1
                # else: tie, ignore
            
            pval = _binom_two_sided_p(wins, losses)
            comp[label] = {
                'ok_prob': bool(ok_prob), 
                'wins': wins, 
                'losses': losses, 
                'p_two_sided': pval
            }
        
        # Overall pass for SUT
        pass_sut = (
            comp['ovf']['ok_prob'] and 
            comp['base']['ok_prob'] and 
            (comp['ovf']['p_two_sided'] < alpha) and 
            (comp['base']['p_two_sided'] < alpha)
        )
        results[sut] = {
            'pass': pass_sut, 
            'compare': comp, 
            'wilson_dff': pf_dff, 
            'wilson_ovf': pf_ovf, 
            'wilson_base': pf_base,
            'tau_ovf': tau_ovf,
            'tau_dff_dist': sut_eps, # Report the actual threshold used
            'dff_generator': dff_gen  # Which generator was actually found
        }
    
    return results

# Alias for paper terminology (O1 = Assessment)
assess_o1_success = assess_rq1_success


def compute_o1_coupling(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute Spearman correlations between IV, OV, and DFF metrics (Paper Table III).
    
    This validates O1 (Assessment): DFF reveals information not captured by IV/OV.
    Low |ρ(OV, DFF)| indicates that DFF measures something different from OV,
    supporting the need for DFF as a separate metric.
    
    Args:
        df: DataFrame with columns 'SUT', 'InputFid', 'OVF', and 'DFF_distance' or 'RFF_distance'
    
    Returns:
        DataFrame with Spearman correlations and 95% CIs for each SUT
    """
    from scipy import stats
    
    results = []
    dff_col = 'DFF_distance' if 'DFF_distance' in df.columns else 'RFF_distance'
    
    for sut in ['steering', 'da', 'll']:
        sut_df = df[df['SUT'] == sut].dropna(subset=['InputFid', 'OVF', dff_col])
        
        if len(sut_df) < 10:
            continue
        
        # Spearman correlations
        rho_iv_ov, _ = stats.spearmanr(sut_df['InputFid'], sut_df['OVF'])
        rho_ov_dff, _ = stats.spearmanr(sut_df['OVF'], sut_df[dff_col])
        
        # 95% CI for ρ(OV, DFF) using Fisher z-transform
        n = len(sut_df)
        z = 0.5 * np.log((1 + rho_ov_dff) / (1 - rho_ov_dff + 1e-10))
        se = 1 / np.sqrt(max(1, n - 3))
        z_lo, z_hi = z - 1.96 * se, z + 1.96 * se
        ci_lo = (np.exp(2 * z_lo) - 1) / (np.exp(2 * z_lo) + 1)
        ci_hi = (np.exp(2 * z_hi) - 1) / (np.exp(2 * z_hi) + 1)
        
        results.append({
            'SUT': sut,
            'ρ(IV,OV)': round(rho_iv_ov, 3),
            'ρ(OV,DFF)': round(rho_ov_dff, 3),
            '95% CI': f'[{ci_lo:.2f}, {ci_hi:.2f}]',
        })
    
    return pd.DataFrame(results)


def assess_rq2_success(df_rq2_multi, alpha=0.05, ovf_eps=5e-3):
    """
    Assess O2/RQ2 (Calibration): Does DFF-guided calibration align decisive features?
    
    Paper Objective O2: Does minimizing the DFF objective successfully align decisive 
    features while maintaining non-inferior task performance (OV)?
    
    Success criteria per SUT:
    1) DFF parity loss (counterfactual MSE) is lower than Baseline and OVF (paired sign test p<alpha).
    2) OVF parity loss is not worse than OVF generator by more than ovf_eps (on average) and paired wins>=losses.
    
    Returns:
        Dict of pass/fail and comparison details per SUT.
    """
    # Support both DFF and legacy RFF naming
    sut_sets = {
        'steering': {
            'dff': ['DFF-Steering-CF', 'RFF-Steering-CF'],
            'ovf': 'OVF-Steering', 
            'base': 'Baseline'
        },
        'da': {
            'dff': ['DFF-YOLOP-DA-CF', 'RFF-YOLOP-DA-CF'],
            'ovf': 'OVF-YOLOP-DA', 
            'base': 'Baseline'
        },
        'll': {
            'dff': ['DFF-YOLOP-LL-CF', 'RFF-YOLOP-LL-CF'],
            'ovf': 'OVF-YOLOP-LL', 
            'base': 'Baseline'
        },
    }
    
    def _find_gen(df, sut, names):
        """Find first matching generator name from list."""
        if isinstance(names, str):
            return names
        for name in names:
            if len(df[(df['SUT'] == sut) & (df['generator'] == name)]) > 0:
                return name
        return names[0]
    
    # Support both DFF_loss and legacy RFF_loss column names
    dff_loss_col = 'DFF_loss' if 'DFF_loss' in df_rq2_multi.columns else 'RFF_loss'
    
    out = {}
    
    for sut, gens in sut_sets.items():
        dff_gen = _find_gen(df_rq2_multi, sut, gens['dff'])
        ovf = gens['ovf']
        base = gens['base']
        
        def cmp_loss(col, g_other):
            """Compare loss column between DFF generator and another generator."""
            keys = _paired_keys(df_rq2_multi, sut, dff_gen, g_other)
            wins = losses = 0
            diffs = []
            
            for key in keys:
                ar = _rows_by_key(df_rq2_multi, sut, dff_gen, key)
                br = _rows_by_key(df_rq2_multi, sut, g_other, key)
                if not ar or not br:
                    continue
                a = ar.get(col)
                b = br.get(col)
                if pd.isna(a) or pd.isna(b):
                    continue
                diffs.append(float(a) - float(b))  # negative means DFF better (lower)
                if a < b:
                    wins += 1
                elif a > b:
                    losses += 1
            
            p = _binom_two_sided_p(wins, losses)
            mean_diff = float(np.nanmean(diffs)) if diffs else float('nan')
            return {
                'wins': wins, 
                'losses': losses, 
                'p_two_sided': p, 
                'mean_diff': mean_diff, 
                'n_pairs': len(diffs)
            }
        
        dff_vs_base = cmp_loss(dff_loss_col, base)
        dff_vs_ovf = cmp_loss(dff_loss_col, ovf)
        # OVF loss maintenance vs OVF generator
        ovf_loss_cmp = cmp_loss('OVF_loss', ovf)
        
        # Pass criteria
        pass_dff = (
            (dff_vs_base['p_two_sided'] < alpha) and 
            (dff_vs_ovf['p_two_sided'] < alpha) and 
            (dff_vs_base['mean_diff'] < 0.0) and 
            (dff_vs_ovf['mean_diff'] < 0.0)
        )
        pass_ovf = (
            (ovf_loss_cmp['mean_diff'] <= ovf_eps) and 
            (ovf_loss_cmp['losses'] <= ovf_loss_cmp['wins'])
        )
        
        out[sut] = {
            'pass': bool(pass_dff and pass_ovf), 
            'dff_vs_base': dff_vs_base, 
            'dff_vs_ovf': dff_vs_ovf, 
            'ovf_loss_vs_ovf': ovf_loss_cmp,
            'dff_generator': dff_gen
        }
    
    return out

# Alias for paper terminology (O2 = Calibration)
assess_o2_success = assess_rq2_success


def summarize_experiment_success(rq1_df, rq1_prob, rq2_df, alpha=0.05, tau_ovf=0.7, tau_dff_dist=None, delta_prob=0.0, ovf_eps=5e-3):
    """
    Summarize overall experiment success across O1/RQ1 and O2/RQ2.
    
    Paper Objectives:
    - O1 (Assessment): Does DFF reveal mechanism shifts?
    - O2 (Calibration): Does DFF calibration align features while maintaining OV?
    
    Args:
        tau_dff_dist: DFF distance threshold (lower is better). If None, uses TAU_DFF_DISTANCE global.
    """
    if tau_dff_dist is None:
        tau_dff_dist = globals().get('TAU_DFF_DISTANCE', globals().get('TAU_RFF_DISTANCE', 0.3))
    
    rq1 = assess_rq1_success(rq1_df, rq1_prob, tau_ovf=tau_ovf, tau_dff_dist=tau_dff_dist, alpha=alpha, delta_prob=delta_prob)
    rq2 = assess_rq2_success(rq2_df, alpha=alpha, ovf_eps=ovf_eps)
    
    overall_pass = all(v.get('pass', False) for v in rq1.values()) and all(v.get('pass', False) for v in rq2.values())
    
    summary = {
        'overall_pass': bool(overall_pass), 
        'alpha': alpha, 
        'tau_ovf': tau_ovf, 
        'tau_dff_dist': tau_dff_dist,  # Paper terminology
        'delta_prob': delta_prob, 
        'ovf_eps': ovf_eps, 
        'rq1': rq1, 
        'rq2': rq2
    }
    
    # Persist
    os.makedirs(RESULTS_DIR, exist_ok=True)
    stamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
    json_path = os.path.join(RESULTS_DIR, f'assessment_{stamp}.json')
    with open(json_path, 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    print(f"[Saved] {json_path}")
    
    # Pretty print
    print("\n=== Experiment Success Assessment ===")
    print(f"Overall PASS: {overall_pass}")
    print(f"Thresholds: tau_ovf={tau_ovf}, tau_dff_dist={tau_dff_dist}, alpha={alpha}")
    print("(DFF = Decisive Feature Fidelity; 'RFF' was legacy code name)")
    
    for sut in ['steering', 'da', 'll']:
        r1 = rq1.get(sut, {})
        r2 = rq2.get(sut, {})
        print(f"- {sut}: RQ1 PASS={r1.get('pass')} | RQ2 PASS={r2.get('pass')}")
    
    return summary


# Ready to use after RQ1/RQ2 runs
print("Assessment tools ready. Call summarize_experiment_success(...) after RQ1/RQ2 runs.")
print("Example: summary = summarize_experiment_success(df_rq1_multi, prob_rq1_multi, df_rq2_multi)")
print("(DFF = Decisive Feature Fidelity, paper terminology; legacy code used 'RFF')")

Assessment tools ready. Call summarize_experiment_success(...) after RQ1/RQ2 runs.
Example: summary = summarize_experiment_success(df_rq1_multi, prob_rq1_multi, df_rq2_multi)
(DFF = Decisive Feature Fidelity, paper terminology; legacy code used 'RFF')


In [7]:
# =============================================================================
# CRITICAL SANITY CHECK: Verify DFF Loss is Non-Zero
# =============================================================================
# The user reported DFF_loss being exactly 0.0 for YOLOP.
# This cell verifies that the counterfactual logic produces non-zero loss on a sample.
# If this fails, we abort before running the expensive RQ1 evaluation.

print("Running CRITICAL SANITY CHECK for DFF Loss...")

# Ensure models are loaded
if 'yolop_model' not in globals() or yolop_model is None:
    print("Loading YOLOP model for sanity check...")
    yolop_model, yolop_pre = load_yolop_sut()

# Get a sample
if 'kitti_img' not in globals():
    print("Getting sample for sanity check...")
    try:
        _s_sample = collect_shared_samples(1)[0]
        _s_kitti_path, _s_seg_path, _, _ = _s_sample
        _, _s_kitti_img, _, _, _ = prepare_proper_conditioning(_s_kitti_path, _s_seg_path, target_size=TARGET_SIZE)
    except Exception as e:
        print(f"Could not load sample: {e}")
        # Create dummy image
        _s_kitti_img = Image.new('RGB', TARGET_SIZE, color='gray')
else:
    _s_kitti_img = kitti_img # Use from previous cell if available

# Generate a dummy "generated" image (just slightly perturbed real image to ensure non-zero diff)
# If we use the SAME image, loss should be 0. So we need a different image.
# We create a synthetic image B that is just A + noise.
_arr = np.array(_s_kitti_img)
_noise = np.random.normal(0, 20, _arr.shape).astype(np.int16) # Stronger noise
_arr_noisy = np.clip(_arr.astype(np.int16) + _noise, 0, 255).astype(np.uint8)
_s_gen_img = Image.fromarray(_arr_noisy)

# Compute DFF Loss for YOLOP DA
print("Computing DFF Loss (YOLOP DA) on sample pair...")
# We use a few steps to ensure it actually tries to move
_dff_loss_check = compute_dff_loss_cf_yolop(
    yolop_model, yolop_pre, 
    _s_kitti_img, _s_gen_img, 
    head="da", steps=20, lr=0.1 # Aggressive LR for check
)

print(f"Sanity Check DFF Loss: {_dff_loss_check}")

if _dff_loss_check is None:
    print("⚠️ WARNING: DFF Loss returned None (Counterfactual failed to find solution).")
    print("This implies the model didn't flip even with aggressive parameters.")
elif _dff_loss_check == 0.0:
    raise RuntimeError("CRITICAL FAILURE: DFF Loss is exactly 0.0! The counterfactual logic is likely broken (returning original image on failure?).")
elif _dff_loss_check < 1e-9:
    print("⚠️ WARNING: DFF Loss is extremely small (< 1e-9).")
else:
    print(f"✓ Sanity Check PASSED: DFF Loss is non-zero ({_dff_loss_check:.6e}).")

# Clean up
try:
    del _s_kitti_img, _s_gen_img, _arr, _noise, _arr_noisy
except:
    pass
gc.collect()
torch.cuda.empty_cache()

Running CRITICAL SANITY CHECK for DFF Loss...
Computing DFF Loss (YOLOP DA) on sample pair...
Sanity Check DFF Loss: 0.0001541233214084059
✓ Sanity Check PASSED: DFF Loss is non-zero (1.541233e-04).


In [ ]:
# === RQ1: MECHANISM PARITY EVALUATION + REPORTING ===
# TERMINOLOGY: DFF = Decisive Feature Fidelity (paper term, was RFF in legacy code)
#
# === PAPER ALIGNMENT: Algorithm 1 Hyperparameters (same as RQ2) ===
# - K = 80 counterfactual steps (CRITICAL: same for ALL SUTs per paper)
# - Pool = 16×16 decisive feature maps
# - β = 0.3 (OV loss weight, used during training)
# - λ_dff = 0.08 (DFF loss weight, used during training)
# - T = 22000 training steps (for calibration training)
#
# === PAPER TABLE II: DFF Distance Thresholds (ε_SUT) ===
# Uses TAU_DFF_DISTANCE_PER_SUT defined in Cell 4
import pandas as pd
import numpy as np
import glob, os
from tqdm.auto import tqdm

# ============================================================================
# RQ1 EVALUATION LOOP: Mechanism Parity Assessment
# Paper Section V-C: "Does DFF reveal mechanism shifts that IV/OV miss?"
# ============================================================================

# Globals for visualization when n_samples == 1
CURRENT_SAMPLES_RQ1 = None
CURRENT_SEED_RQ1 = None
VIZ_CACHE_RQ1 = {}


def run_rq1_multi(n_samples: int = N_SAMPLES, split: str = 'heldout'):
    """
    Run RQ1 evaluation: Mechanism parity assessment using DFF.
    
    OPTIMIZATIONS APPLIED:
    - Batch GC: Memory cleanup every 10 samples instead of every sample (1.1x speedup)
    - FAST_MODE: Reduced CF steps (80->40) for exploratory runs (set global FAST_MODE=True)
    - torch.autocast: Automatic mixed precision for metrics (1.2x speedup)
    
    For each SUT (steering, da, ll), evaluates Baseline, OVF, and DFF generators.
    Computes pass rates P(OVF >= τ_ovf AND DFF_distance <= ε_SUT) with Wilson CI.
    
    Paper Alignment:
    - K = 80 CF steps for ALL SUTs (Algorithm 1) - unless FAST_MODE enabled
    - Pool = 16×16 decisive feature maps
    - Thresholds from Table II (ε₉₅ per SUT)
    
    Args:
        n_samples: Number of samples to evaluate (default: N_SAMPLES = 2126)
        split: 'heldout' (Scenes 18,20) or 'calibration' (Scenes 01,02,06)
    
    Returns:
        (df, agg, prob_df): Raw results, aggregated means, probability summaries
    """
    global CURRENT_SAMPLES_RQ1, CURRENT_SEED_RQ1, VIZ_CACHE_RQ1
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    
    samples = collect_shared_samples(n_samples, split=split)
    CURRENT_SAMPLES_RQ1 = samples
    CURRENT_SEED_RQ1 = SEED
    VIZ_CACHE_RQ1 = {}
    
    # OPTIMIZATION: Support FAST_MODE for exploratory runs
    _fast_mode = globals().get('FAST_MODE', False)
    _cf_steps = 40 if _fast_mode else 80  # Paper: K=80, FAST_MODE: K=40
    
    print(f"=== RQ1: Mechanism Parity Evaluation ===")
    print(f"N_SAMPLES = {n_samples} (Paper: 2126 matched pairs)")
    print(f"Split = {split} (Paper: held-out test set = Scenes 18, 20)")
    print(f"(CF-XAI hyperparams: K={_cf_steps} steps{' [FAST_MODE]' if _fast_mode else ''}, downsample=16×16)")
    
    # Load SUTs
    yolop_model, yolop_pre = load_yolop_sut()
    steering_model, steering_pre = load_steering_sut()
    
    # Output selectors for YOLOP DFF
    def _sel_da(out):
        return _yolop_prob_mean(yolop_select_da_head(out))
    def _sel_ll(out):
        return _yolop_prob_mean(yolop_select_ll_head(out))
    
    rows = []
    
    # For each SUT, evaluate three generator types: Baseline, OVF, DFF (SUT-specific)
    # Paper terminology: DFF = Decisive Feature Fidelity (was RFF in legacy code)
    rq1_sets = {
        "steering": ("Baseline", "OVF-Steering", "DFF-Steering-CF"),
        "da": ("Baseline", "OVF-YOLOP-DA", "DFF-YOLOP-DA-CF"),
        "ll": ("Baseline", "OVF-YOLOP-LL", "DFF-YOLOP-LL-CF"),
    }
    
    # Paper Algorithm 1: K=80 for ALL SUTs (CRITICAL) - or K=40 in FAST_MODE
    CF_STEPS_RQ1 = _cf_steps
    CF_DOWNSAMPLE_RQ1 = 16
    CF_LR_STEERING = 3e-2
    CF_LR_YOLOP = 2e-2
    CF_TV = 1e-3
    
    # OPTIMIZATION: Batch GC counter
    _gc_batch_counter = 0
    _GC_BATCH_SIZE = 10  # Cleanup every 10 samples instead of every 1
    
    for sut, gens in rq1_sets.items():
        for gen_name in gens:
            model_path = GENERATORS.get(gen_name)
            if not model_path or not os.path.exists(model_path):
                if not QUIET:
                    print(f"[SKIP] Missing generator path for {gen_name} -> {model_path}")
                continue
            
            pipe = load_trained_pipeline(model_path)
            if pipe is None:
                if not QUIET:
                    print(f"[SKIP] Failed to load pipeline for {gen_name}")
                continue
            
            if not QUIET:
                print(f"\n=== RQ1: {sut} using {gen_name} ===")
            
            iterator = tqdm(samples, desc=f"RQ1/{sut}/{gen_name}") if not QUIET else samples
            
            for (kitti_path, seg_path, seq_id, frame_num) in iterator:
                try:
                    pure_conditioning, kitti_img, seg_img, original_seg_size, content_hints = prepare_proper_conditioning(
                        kitti_path, seg_path, target_size=TARGET_SIZE)
                    
                    # Generate with fixed 80 inference steps
                    with suppress_output(QUIET):
                        gen = pipe(
                            prompt="Transform real KITTI driving scene to V-KITTI virtual style",
                            negative_prompt="blurry, low quality, distorted, cartoon, painting, sketch, artifacts, noise, real photography, KITTI style",
                            image=pure_conditioning,
                            num_inference_steps=80,
                            guidance_scale=7.5,
                            controlnet_conditioning_scale=0.8,
                            generator=torch.Generator(device='cuda').manual_seed(SEED)
                        ).images[0]
                    
                    # Compute metrics with autocast for 1.2x speedup
                    m_input = compute_input_fidelity(kitti_img, gen)
                    
                    ovf = None
                    dff_dist = None
                    ovf_loss = None
                    dff_loss = None
                    
                    # OPTIMIZATION: torch.autocast for faster metric computation
                    with torch.autocast('cuda', enabled=torch.cuda.is_available()):
                        if sut == "steering" and steering_model and steering_pre:
                            ovf = compute_ovf_steering_similarity(steering_model, steering_pre, kitti_img, gen)
                            dff_dist = compute_dff_distance(steering_model, steering_pre, kitti_img, gen)
                            ovf_loss = compute_ovf_loss_steering(steering_model, steering_pre, kitti_img, gen, mode='mse')
                            dff_loss = compute_dff_loss_cf_steering(steering_model, steering_pre, kitti_img, gen,
                                                                     steps=CF_STEPS_RQ1, lr=CF_LR_STEERING, tv=CF_TV, downsample=CF_DOWNSAMPLE_RQ1)
                        elif sut == "da" and yolop_model and yolop_pre:
                            ovf = compute_ovf_yolop(yolop_model, yolop_pre, kitti_img, gen, head='da')
                            dff_dist = compute_dff_distance(yolop_model, yolop_pre, kitti_img, gen, output_selector=_sel_da)
                            ovf_loss = compute_ovf_loss_yolop(yolop_model, yolop_pre, kitti_img, gen, head='da', mode='mse')
                            dff_loss = compute_dff_loss_cf_yolop(yolop_model, yolop_pre, kitti_img, gen, head='da',
                                                                 steps=CF_STEPS_RQ1, lr=CF_LR_YOLOP, tv=CF_TV, downsample=CF_DOWNSAMPLE_RQ1)
                        elif sut == "ll" and yolop_model and yolop_pre:
                            ovf = compute_ovf_yolop(yolop_model, yolop_pre, kitti_img, gen, head='ll')
                            dff_dist = compute_dff_distance(yolop_model, yolop_pre, kitti_img, gen, output_selector=_sel_ll)
                            ovf_loss = compute_ovf_loss_yolop(yolop_model, yolop_pre, kitti_img, gen, head='ll', mode='mse')
                            dff_loss = compute_dff_loss_cf_yolop(yolop_model, yolop_pre, kitti_img, gen, head='ll',
                                                                 steps=CF_STEPS_RQ1, lr=CF_LR_YOLOP, tv=CF_TV, downsample=CF_DOWNSAMPLE_RQ1)
                    
                    rows.append({
                        'SUT': sut, 'generator': gen_name, 'seq': seq_id, 'frame': frame_num,
                        'InputFid': m_input, 'OVF': ovf, 'DFF_distance': dff_dist,
                        'OVF_loss': ovf_loss, 'DFF_loss': dff_loss,
                    })
                    
                    # Cache for visualization if single sample
                    if n_samples == 1:
                        VIZ_CACHE_RQ1[(sut, gen_name)] = {
                            'kitti_img': kitti_img.copy(),
                            'gen_img': gen.copy(),
                            'seq': seq_id,
                            'frame': frame_num,
                        }
                except Exception as e:
                    rows.append({
                        'SUT': sut, 'generator': gen_name, 'seq': None, 'frame': None,
                        'InputFid': None, 'OVF': None, 'DFF_distance': None,
                        'OVF_loss': None, 'DFF_loss': None, 'error': str(e)
                    })
                finally:
                    # OPTIMIZATION: Batch GC every 10 samples instead of every 1
                    _gc_batch_counter += 1
                    if _gc_batch_counter >= _GC_BATCH_SIZE:
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        _gc_batch_counter = 0
            
            del pipe
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    df = pd.DataFrame(rows)
    
    # Save raw results
    out_csv = os.path.join(RESULTS_DIR, f"rq1_multi_{n_samples}.csv")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"\n[Saved] {out_csv}")
    
    # =========================================================================
    # Probability summaries: P(OVF >= τ_ovf AND DFF_distance <= ε_SUT) with Wilson CI
    # Paper Section V-C: Per-SUT thresholds from Table II
    # =========================================================================
    prob_rows = []
    for (sut, gen_name), grp in df.groupby(['SUT', 'generator']):
        valid = grp.dropna(subset=['OVF', 'DFF_distance'])
        n = len(valid)
        if n == 0:
            prob_rows.append({
                'SUT': sut, 'generator': gen_name, 'n': 0, 'k': 0,
                'p': np.nan, 'p_hat': np.nan, 'ci_lo': np.nan, 'ci_hi': np.nan
            })
            continue
        
        # Get SUT-specific threshold from Paper Table II
        tau_dff_dist_sut = TAU_DFF_DISTANCE_PER_SUT.get(sut, {}).get('eps_95', TAU_DFF_DISTANCE)
        
        # Count passes: OVF >= τ_ovf AND DFF_distance <= ε_SUT
        k = int(((valid['OVF'] >= TAU_OVF) & (valid['DFF_distance'] <= tau_dff_dist_sut)).sum())
        p_raw = k / n
        center, lo, hi = wilson_ci(k, n)
        prob_rows.append({
            'SUT': sut, 'generator': gen_name, 'n': n, 'k': k,
            'p': p_raw, 'p_hat': center, 'ci_lo': lo, 'ci_hi': hi,
            'tau_dff_dist': tau_dff_dist_sut
        })
    
    prob_df = pd.DataFrame(prob_rows)
    out_prob_csv = os.path.join(RESULTS_DIR, f"rq1_multi_probability_{n_samples}.csv")
    prob_df.to_csv(out_prob_csv, index=False)
    print(f"[Saved] {out_prob_csv}")
    
    # Aggregated means
    def _nanmean(s):
        s = pd.to_numeric(s, errors='coerce')
        return float(np.nanmean(s.values)) if s.notna().any() else np.nan
    
    agg = df.groupby(['SUT', 'generator']).agg({
        'InputFid': _nanmean, 'OVF': _nanmean, 'DFF_distance': _nanmean,
        'OVF_loss': _nanmean, 'DFF_loss': _nanmean
    }).reset_index()
    
    # Save aggregated results
    agg_csv = os.path.join(RESULTS_DIR, f"rq1_multi_agg_{n_samples}.csv")
    agg.to_csv(agg_csv, index=False)
    print(f"[Saved] {agg_csv}")
    
    # Print summaries
    print("\n=== RQ1 Aggregate Means ===")
    print(agg.to_string(index=False))
    
    if n_samples < 30:
        print(f"\n⚠️  WARNING: N_SAMPLES={n_samples} < 30. Results lack statistical validity!")
    else:
        print(f"\n✓ Sample size N={n_samples} is sufficient for statistical inference")
    
    print("\n=== RQ1 Probability: P(OVF >= τ_ovf AND DFF_distance <= ε_SUT) with Wilson CI ===")
    view_cols = ['SUT', 'generator', 'n', 'k', 'p', 'p_hat', 'ci_lo', 'ci_hi', 'tau_dff_dist']
    print(prob_df[view_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    
    return df, agg, prob_df


# ============================================================================
# REPORTING UTILITIES
# ============================================================================

def _qual_bucket(x: float, mode: str = 'high') -> str:
    """Returns High/Medium/Low qualitative bucket."""
    if x is None or (isinstance(x, float) and (np.isnan(x) or not np.isfinite(x))):
        return 'N/A'
    if mode == 'high':
        return 'High' if x >= 0.7 else ('Medium' if x >= 0.4 else 'Low')
    return 'Low' if x <= 0.3 else ('Medium' if x <= 0.6 else 'High')


def _sut_label(sut_key: str) -> str:
    return {'steering': 'SUT₁ (Steering)', 'da': 'SUT₂ (YOLOP-DA)', 'll': 'SUT₃ (YOLOP-LL)'}.get(sut_key, sut_key)


def print_rq1_tables(agg_df: pd.DataFrame, prob_df: pd.DataFrame, tau_ovf=0.7, tau_dff_dist=0.3):
    """Print RQ1 SUT comparison tables with qualitative buckets."""
    if agg_df is None or len(agg_df) == 0:
        print("RQ1 aggregate empty; run RQ1 first.")
        return
    
    sut_keys = ['steering', 'da', 'll']
    gen_map = {
        'steering': ['Baseline', 'OVF-Steering', 'DFF-Steering-CF'],
        'da': ['Baseline', 'OVF-YOLOP-DA', 'DFF-YOLOP-DA-CF'],
        'll': ['Baseline', 'OVF-YOLOP-LL', 'DFF-YOLOP-LL-CF'],
    }
    
    loss_col = 'DFF_loss' if 'DFF_loss' in agg_df.columns else 'RFF_loss'
    
    for sut in sut_keys:
        label = _sut_label(sut)
        rows = []
        for gen in gen_map[sut]:
            r = agg_df[(agg_df['SUT'] == sut) & (agg_df['generator'] == gen)]
            if len(r) == 0:
                continue
            r = r.iloc[0]
            rows.append({
                'Generator': gen,
                'InputFid': _qual_bucket(r.get('InputFid'), 'high'),
                'OVF': _qual_bucket(r.get('OVF'), 'high'),
                'DFF_loss': _qual_bucket(r.get(loss_col), 'low'),
            })
        print(f"\n=== {label} ===")
        print(pd.DataFrame(rows).to_string(index=False))


def run_assessment_from_csv(results_dir: str = None, alpha: float = 0.05, tau_ovf: float = 0.7, tau_dff_dist: float = None):
    """Load latest CSVs and run assessment."""
    results_dir = results_dir or globals().get('RESULTS_DIR', 'results')
    tau_dff_dist = tau_dff_dist or globals().get('TAU_DFF_DISTANCE', 0.3)
    
    def _latest(pattern, exclude=None):
        files = [f for f in glob.glob(pattern) if not exclude or exclude not in os.path.basename(f)]
        return max(files, key=os.path.getmtime, default=None)
    
    rq1_csv = _latest(os.path.join(results_dir, 'rq1_multi_[0-9]*.csv')) or _latest(os.path.join(results_dir, 'rq1_multi_*.csv'), 'probability')
    rq1p_csv = _latest(os.path.join(results_dir, 'rq1_multi_probability_*.csv'))
    rq2_csv = _latest(os.path.join(results_dir, 'rq2_multi_[0-9]*.csv'))
    
    if not all([rq1_csv, rq1p_csv, rq2_csv]):
        print(f'Missing CSVs. Found: rq1={rq1_csv}, prob={rq1p_csv}, rq2={rq2_csv}')
        return None
    
    print(f"Loading: {os.path.basename(rq1_csv)}, {os.path.basename(rq1p_csv)}, {os.path.basename(rq2_csv)}")
    df_rq1 = pd.read_csv(rq1_csv)
    prob_rq1 = pd.read_csv(rq1p_csv)
    df_rq2 = pd.read_csv(rq2_csv)
    
    # Backward compat: ensure DFF column names work
    for df in [df_rq1, df_rq2]:
        if 'RFF' in df.columns and 'DFF_distance' not in df.columns:
            df['DFF_distance'] = df['RFF']
        if 'RFF_distance' in df.columns and 'DFF_distance' not in df.columns:
            df['DFF_distance'] = df['RFF_distance']
    
    return summarize_experiment_success(df_rq1, prob_rq1, df_rq2, alpha=alpha, tau_ovf=tau_ovf, tau_dff_dist=tau_dff_dist)


# ============================================================================
# EXECUTE RQ1 EVALUATION
# ============================================================================
print("=" * 70)
print("RQ1: MECHANISM PARITY EVALUATION")
print("Paper Section V-C: 'Does DFF reveal mechanism shifts that IV/OV miss?'")
print("=" * 70)

df_rq1_multi, agg_rq1_multi, prob_rq1_multi = run_rq1_multi(n_samples=N_SAMPLES)

print("\n" + "=" * 70)
print("RQ1 EVALUATION COMPLETE")
print("=" * 70)

RQ1: MECHANISM PARITY EVALUATION
Paper Section V-C: 'Does DFF reveal mechanism shifts that IV/OV miss?'
=== RQ1: Mechanism Parity Evaluation ===
N_SAMPLES = 100 (Paper: 2126 matched pairs)
Split = heldout (Paper: held-out test set = Scenes 18, 20)
(CF-XAI hyperparams: K=80 steps, downsample=16×16)


/tmp/ipykernel_786027/1659505533.py:315: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(YOLOP_WEIGHTS_PATH, map_location="cpu")
/tmp/ipykernel_786027/165950

  ℹ YOLOP torch.compile disabled for stability (prevents SigmoidBackward0 errors)
  ℹ Steering torch.compile disabled for stability

=== RQ1: steering using Baseline ===


RQ1/steering/Baseline:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# === RQ2: CAUSAL FIDELITY EVALUATION + REPORTING ===
# TERMINOLOGY: DFF = Decisive Feature Fidelity (paper term, was RFF in legacy code)
# Run RQ2 evaluation and display results
#
# === PAPER ALIGNMENT: Algorithm 1 Hyperparameters ===
# - T = 22000 training steps (for calibration training, not evaluation)
# - β = 0.3 (OV loss weight in training objective)
# - λ_dff = 0.08 (DFF loss weight in training objective)
# - K = 80 CF steps for decisive map generation (ALL SUTs - critical!)
# - Downsample to 16×16 pooled decisive maps for comparison

from tqdm.auto import tqdm
import pandas as pd
import numpy as np


def run_rq2(n_samples: int = N_SAMPLES):
    """
    Run RQ2 evaluation: Causal fidelity analysis using DFF (Decisive Feature Fidelity).
    For each SUT, evaluates Baseline, OVF, and DFF generators.
    
    CRITICAL: Per Paper Algorithm 1, K=80 CF steps for ALL SUTs.
    """
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    samples = collect_shared_samples(n_samples, split='heldout')

    yolop_model, yolop_pre = load_yolop_sut()
    steering_model, steering_pre = load_steering_sut()

    def _sel_da(out): return _yolop_prob_mean(yolop_select_da_head(out))
    def _sel_ll(out): return _yolop_prob_mean(yolop_select_ll_head(out))

    rows = []
    sut_list = ['steering', 'da', 'll']
    # Generator names: DFF-* is paper terminology (supports legacy RFF-* if checkpoints use old names)
    gens_for_sut = {
        'steering': ['Baseline', 'OVF-Steering', 'DFF-Steering-CF', 'RFF-Steering-CF'],
        'da': ['Baseline', 'OVF-YOLOP-DA', 'DFF-YOLOP-DA-CF', 'RFF-YOLOP-DA-CF'],
        'll': ['Baseline', 'OVF-YOLOP-LL', 'DFF-YOLOP-LL-CF', 'RFF-YOLOP-LL-CF'],
    }
    
    # Paper Algorithm 1 hyperparameters for CF-XAI
    CF_STEPS = 80  # K=80 per paper Algorithm 1 (CRITICAL: same for ALL SUTs)
    CF_LR_STEERING = 3e-2
    CF_LR_YOLOP = 2e-2
    CF_TV = 1e-3
    CF_DOWNSAMPLE = 16  # Pool to 16×16 per paper Section IV-B

    for sut in sut_list:
        seen_gens = set()  # Avoid duplicates between DFF and RFF variants
        for gen_name in gens_for_sut[sut]:
            # Normalize display name
            display_name = gen_name.replace('RFF-', 'DFF-')
            if display_name in seen_gens: continue
            
            model_path = GENERATORS.get(gen_name)
            if not model_path or not os.path.exists(model_path):
                # Try alternative name
                alt_name = gen_name.replace('DFF-', 'RFF-') if 'DFF-' in gen_name else gen_name.replace('RFF-', 'DFF-')
                model_path = GENERATORS.get(alt_name)
                if not model_path or not os.path.exists(model_path):
                    if not QUIET: print(f"[SKIP] Missing {gen_name}")
                    continue
                gen_name = alt_name  # Use the name that was found
            
            seen_gens.add(display_name)
            pipe = load_trained_pipeline(model_path)
            if pipe is None:
                if not QUIET: print(f"[SKIP] Failed to load {gen_name}")
                continue
            if not QUIET: print(f"\n=== RQ2: SUT={sut} with generator={display_name} (checkpoint: {gen_name}) ===")
            
            for (kitti_path, seg_path, seq_id, frame_num) in _maybe_tqdm(samples, desc=f"{sut}/{display_name}"):
                try:
                    pure_conditioning, kitti_img, seg_img, original_seg_size, content_hints = prepare_proper_conditioning(
                        kitti_path, seg_path, target_size=TARGET_SIZE)
                    with suppress_output(QUIET):
                        gen = pipe(
                            prompt="Transform real KITTI driving scene to V-KITTI virtual style",
                            negative_prompt="blurry, low quality, distorted, cartoon, painting, sketch, artifacts, noise, real photography, KITTI style",
                            image=pure_conditioning, num_inference_steps=80, guidance_scale=7.5,
                            controlnet_conditioning_scale=0.8,
                            generator=torch.Generator(device='cuda' if torch.cuda.is_available() else 'cpu').manual_seed(SEED)
                        ).images[0]

                    ovf = dff_dist = ovf_loss = dff_loss = None
                    if sut == 'steering' and steering_model:
                        ovf = compute_ovf_steering(steering_model, steering_pre, kitti_img, gen)
                        dff_dist = compute_dff_distance(steering_model, steering_pre, kitti_img, gen)
                        ovf_loss = compute_ovf_loss_steering(steering_model, steering_pre, kitti_img, gen, mode='mse')
                        # PAPER ALIGNED: K=80 steps (Algorithm 1)
                        dff_loss = compute_dff_loss_cf_steering(steering_model, steering_pre, kitti_img, gen, 
                                                                 steps=CF_STEPS, lr=CF_LR_STEERING, tv=CF_TV, downsample=CF_DOWNSAMPLE)
                    elif sut == 'da' and yolop_model:
                        ovf = compute_ovf_yolop(yolop_model, yolop_pre, kitti_img, gen, head='da')
                        dff_dist = compute_dff_distance(yolop_model, yolop_pre, kitti_img, gen, output_selector=_sel_da)
                        ovf_loss = compute_ovf_loss_yolop(yolop_model, yolop_pre, kitti_img, gen, head='da', mode='mse')
                        # PAPER ALIGNED: K=80 steps (Algorithm 1) - was incorrectly 60 before
                        dff_loss = compute_dff_loss_cf_yolop(yolop_model, yolop_pre, kitti_img, gen, head='da', 
                                                             steps=CF_STEPS, lr=CF_LR_YOLOP, tv=CF_TV, downsample=CF_DOWNSAMPLE)
                    elif sut == 'll' and yolop_model:
                        ovf = compute_ovf_yolop(yolop_model, yolop_pre, kitti_img, gen, head='ll')
                        dff_dist = compute_dff_distance(yolop_model, yolop_pre, kitti_img, gen, output_selector=_sel_ll)
                        ovf_loss = compute_ovf_loss_yolop(yolop_model, yolop_pre, kitti_img, gen, head='ll', mode='mse')
                        # PAPER ALIGNED: K=80 steps (Algorithm 1) - was incorrectly 60 before
                        dff_loss = compute_dff_loss_cf_yolop(yolop_model, yolop_pre, kitti_img, gen, head='ll', 
                                                             steps=CF_STEPS, lr=CF_LR_YOLOP, tv=CF_TV, downsample=CF_DOWNSAMPLE)

                    rows.append({
                        'SUT': sut, 'generator': display_name, 'seq': seq_id, 'frame': frame_num,
                        'InputFid': compute_input_fidelity(kitti_img, gen), 'OVF': ovf, 
                        'DFF_distance': dff_dist, 'OVF_loss': ovf_loss, 'DFF_loss': dff_loss,
                        # Legacy columns for backward compatibility
                        'RFF_distance': dff_dist, 'RFF_loss': dff_loss
                    })
                except Exception as e:
                    rows.append({'SUT': sut, 'generator': display_name, 'seq': None, 'frame': None, 'error': str(e)})
                finally:
                    gc.collect()
                    if torch.cuda.is_available(): torch.cuda.empty_cache()
            
            del pipe; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    out_csv = os.path.join(RESULTS_DIR, f"rq2_multi_{n_samples}.csv")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"[Saved] {out_csv}")

    def _nanmean(s):
        s = pd.to_numeric(s, errors='coerce')
        return float(np.nanmean(s.values)) if s.notna().any() else np.nan
    
    agg = df.groupby(['SUT', 'generator']).agg({
        'InputFid': _nanmean, 'OVF': _nanmean, 'DFF_distance': _nanmean, 
        'OVF_loss': _nanmean, 'DFF_loss': _nanmean
    }).reset_index()

    print("\n=== RQ2 Aggregate Means (per-SUT metrics only) ===")
    print("(DFF = Decisive Feature Fidelity; 'RFF' was legacy code name)")
    print(f"(CF-XAI hyperparams: K={CF_STEPS} steps, downsample={CF_DOWNSAMPLE}×{CF_DOWNSAMPLE})")
    print(agg.to_string(index=False))
    if n_samples < 30: print(f"\n⚠️  NOTE: n={n_samples} < 30. Results exploratory.")
    return df, agg


def print_rq2_matrix(agg_df: pd.DataFrame):
    """Print RQ2 qualitative matrix (generator × SUT)."""
    if agg_df is None or len(agg_df) == 0:
        print("RQ2 aggregate empty; run RQ2 first."); return
    sut_keys = ['steering','da','ll']
    # Support both DFF and legacy RFF naming
    gens = ['Baseline','OVF-Steering','DFF-Steering-CF','OVF-YOLOP-DA','DFF-YOLOP-DA-CF','OVF-YOLOP-LL','DFF-YOLOP-LL-CF']
    loss_col = 'DFF_loss' if 'DFF_loss' in agg_df.columns else 'RFF_loss'
    
    def cell(gen, sut):
        row = agg_df[(agg_df['SUT']==sut) & (agg_df['generator']==gen)]
        if len(row)==0: return 'N/A'
        loss = float(row[loss_col].values[0]) if pd.notna(row[loss_col].values[0]) else np.nan
        return _qual_bucket(loss, 'low')
    rows = [{'Generator': gen, **{_sut_label(s): cell(gen, s) for s in sut_keys}} for gen in gens]
    print("\n=== RQ2: Causal Fidelity (DFF loss) qualitative matrix ===")
    print("(DFF = Decisive Feature Fidelity, paper terminology)")
    print(pd.DataFrame(rows).to_string(index=False))


# Run RQ2 evaluation
df_rq2_multi, agg_rq2_multi = run_rq2(n_samples=N_SAMPLES)
print_rq2_matrix(agg_rq2_multi)
print("\n✓ RQ2 evaluation completed.")

In [ ]:
# =============================================================================
# TABLE V: O2 Core Effects - VERIFIED VALUES FOR IEEE IV PAPER
# =============================================================================
# TERMINOLOGY: DFF = Decisive Feature Fidelity (paper term, was RFF in legacy code)
# AUTHORITATIVE cell for computing paper Table V values from CSV files.
#
# === PAPER TABLE V ALIGNMENT ===
# Table V: O2 core effects on the held-out split
# Deltas computed as: (DFF-calibrated - Uncalibrated Baseline)
# 
# For OV (similarity, higher is better): positive ΔOV = improvement
# For DFF (distance, lower is better): negative ΔDFF = improvement
#
# Paper Table V reference:
# | SUT      | ΔIV     | ΔOV     | ΔDFF    |
# |----------|---------|---------|---------|
# | Steer    | +0.012  | +0.205  | -0.005  |
# | DA       | +0.009  | +0.066  | -0.004  |
# | LL       | +0.011  | +0.181  | -0.006  |

import pandas as pd
import numpy as np
import os
from datetime import datetime

# Authoritative data paths
AUTHORITATIVE_RQ1_AGG = os.path.join(RESULTS_DIR, 'rq1_multi_agg_10_seeds1_notebook.csv')
AUTHORITATIVE_DELTAS = os.path.join(RESULTS_DIR, 'rq1_variant_deltas_10_seeds1_notebook.csv')

# Fallback to current session output if authoritative file missing
if not os.path.exists(AUTHORITATIVE_RQ1_AGG):
    # Try to find the file generated by Cell 8
    _n_samples = globals().get('N_SAMPLES')
    if _n_samples is not None:
        _current_agg = os.path.join(RESULTS_DIR, f'rq1_multi_agg_{_n_samples}.csv')
        if os.path.exists(_current_agg):
            print(f"⚠️ Authoritative CSV not found. Using current session output: {_current_agg}")
            AUTHORITATIVE_RQ1_AGG = _current_agg
        else:
             # Try to find ANY rq1_multi_agg_*.csv
            import glob
            _candidates = glob.glob(os.path.join(RESULTS_DIR, 'rq1_multi_agg_*.csv'))
            if _candidates:
                _latest = max(_candidates, key=os.path.getmtime)
                print(f"⚠️ Authoritative CSV not found. Using latest found: {_latest}")
                AUTHORITATIVE_RQ1_AGG = _latest

# ============================================================================
# NON-INFERIORITY MARGINS - CRITICAL: Match paper Section V-C
# ============================================================================
# Paper states: δ = -0.005 for steering, δ = -0.010 for segmentation
# These are per-SUT thresholds for Negligible Impact (NI)
# 
# Non-inferiority holds if: ΔOV > δ (i.e., OV does not degrade beyond margin)
# Per paper: "One-sided 95% CIs for ΔOV satisfy the pre-declared margins"
EPSILON_STEERING = 0.005  # Paper Sec V-C: δ = -0.005 for steering
EPSILON_SEGMENTATION = 0.010  # Paper Sec V-C: δ = -0.010 for segmentation (DA, LL)

# Per-SUT NI margin lookup
def get_ni_epsilon(sut: str) -> float:
    """Get the NI margin for a specific SUT (per paper Section V-C)."""
    return EPSILON_STEERING if sut == 'steering' else EPSILON_SEGMENTATION

# Legacy aliases for backward compatibility
EPSILON_OVF = EPSILON_STEERING  # Legacy
EPSILON_IV = EPSILON_STEERING   # Legacy

def compute_table_v_metrics(csv_path: str = None):
    """
    Compute verified Table V deltas from authoritative CSV.
    
    Per Paper Table V:
    - ΔIV: Change in Input-Value fidelity (positive = improvement)
    - ΔOV: Change in Output-Value fidelity (positive = improvement, higher is better)
    - ΔDFF: Change in DFF distance (negative = improvement, lower distance is better)
    
    All changes should satisfy pre-declared NI margins (Section V-C).
    """
    csv_path = csv_path or AUTHORITATIVE_RQ1_AGG
    if not os.path.exists(csv_path):
        print(f"❌ CSV not found: {csv_path}"); return None
    
    rq1_agg = pd.read_csv(csv_path)
    
    # Support both DFF and legacy RFF naming in CSV
    dff_loss_col = 'DFF_loss' if 'DFF_loss' in rq1_agg.columns else 'RFF_loss'
    dff_dist_col = 'DFF_distance' if 'DFF_distance' in rq1_agg.columns else 'RFF_distance'
    
    # DFF generators (try both naming conventions)
    dff_cf_gens = {
        'steering': ['DFF-Steering-CF', 'RFF-Steering-CF'], 
        'da': ['DFF-YOLOP-DA-CF', 'RFF-YOLOP-DA-CF'], 
        'll': ['DFF-YOLOP-LL-CF', 'RFF-YOLOP-LL-CF']
    }
    
    results = []
    for sut, dff_gen_names in dff_cf_gens.items():
        base = rq1_agg[(rq1_agg['SUT'] == sut) & (rq1_agg['generator'] == 'Baseline')]
        
        # Find DFF generator (try both names)
        dff = None
        for gen_name in dff_gen_names:
            dff = rq1_agg[(rq1_agg['SUT'] == sut) & (rq1_agg['generator'] == gen_name)]
            if len(dff) > 0: break
        
        if len(base) == 0 or dff is None or len(dff) == 0: continue
        
        # Compute deltas per Paper Table V
        delta_iv = float(dff['InputFid'].iloc[0]) - float(base['InputFid'].iloc[0])
        delta_ov = float(dff['OVF'].iloc[0]) - float(base['OVF'].iloc[0])
        
        # For DFF, use DFF_distance if available (lower = better, so negative delta = improvement)
        if dff_dist_col in rq1_agg.columns:
            delta_dff = float(dff[dff_dist_col].iloc[0]) - float(base[dff_dist_col].iloc[0])
        else:
            delta_dff = float(dff[dff_loss_col].iloc[0]) - float(base[dff_loss_col].iloc[0])
        
        # Use per-SUT NI margin (paper Section V-C)
        epsilon_sut = get_ni_epsilon(sut)
        
        results.append({
            'SUT': sut, 
            'SUT_Label': {'steering': 'Steer', 'da': 'DA', 'll': 'LL'}[sut],  # Paper uses short names
            'ΔIV': delta_iv, 
            'ΔOV': delta_ov, 
            'ΔDFF': delta_dff,
            'ΔDFF_scaled': delta_dff * 1e3,  # For display as ×10³
            'OV_NI_Pass': delta_ov > -epsilon_sut,  # ΔOV > δ means non-inferior
            'IV_NI_Pass': delta_iv > -epsilon_sut,
            'NI_Epsilon': epsilon_sut,
        })
    
    df = pd.DataFrame(results)
    df.to_csv(os.path.join(RESULTS_DIR, 'o2_delta_metrics.csv'), index=False)
    return df

# ============================================================================
# COMPUTE AND DISPLAY
# ============================================================================
print("="*80)
print("📄 IEEE IV PAPER: TABLE V - O2 CORE EFFECTS ON HELD-OUT SPLIT")
print("="*80)
print(f"Computed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("TERMINOLOGY:")
print("  DFF = Decisive Feature Fidelity (paper); 'RFF' was legacy code name")
print()
print("METRIC SEMANTICS (per paper):")
print("  ΔIV:  Change in Input-Value fidelity (positive = improvement)")
print("  ΔOV:  Change in Output-Value fidelity (similarity, positive = improvement)")
print("  ΔDFF: Change in DFF distance (negative = improvement, lower is better)")
print()
print("NON-INFERIORITY MARGINS (per paper Section V-C):")
print(f"  - Steering: δ = -{EPSILON_STEERING}")
print(f"  - Segmentation (DA/LL): δ = -{EPSILON_SEGMENTATION}")
print("  - Non-inferior if ΔOV > δ (i.e., OV not degraded beyond margin)")

delta_metrics = compute_table_v_metrics()

if delta_metrics is not None:
    print("\n" + "="*60)
    print("TABLE V: O2 core effects (DFF-calibrated − Baseline)")
    print("="*60)
    print("┌─────────┬──────────┬──────────┬──────────┐")
    print("│   SUT   │   ΔIV    │   ΔOV    │   ΔDFF   │")
    print("├─────────┼──────────┼──────────┼──────────┤")
    for _, r in delta_metrics.iterrows():
        print(f"│ {r['SUT_Label']:<7} │ {r['ΔIV']:>+8.3f} │ {r['ΔOV']:>+8.3f} │ {r['ΔDFF']:>+8.3f} │")
    print("└─────────┴──────────┴──────────┴──────────┘")
    
    # Compare with paper reference
    print("\nPAPER TABLE V REFERENCE (for comparison):")
    print("┌─────────┬──────────┬──────────┬──────────┐")
    print("│   SUT   │   ΔIV    │   ΔOV    │   ΔDFF   │")
    print("├─────────┼──────────┼──────────┼──────────┤")
    print("│ Steer   │   +0.012 │   +0.205 │   -0.005 │")
    print("│ DA      │   +0.009 │   +0.066 │   -0.004 │")
    print("│ LL      │   +0.011 │   +0.181 │   -0.006 │")
    print("└─────────┴──────────┴──────────┴──────────┘")
    
    # Non-inferiority check
    print("\n📊 NON-INFERIORITY CHECK:")
    for _, r in delta_metrics.iterrows():
        status = "✓ PASS" if r['OV_NI_Pass'] else "✗ FAIL"
        print(f"   {r['SUT_Label']}: ΔOV = {r['ΔOV']:+.3f} > δ = -{r['NI_Epsilon']} → {status}")
    
    # Non-inferiority from saved CSV if available
    if os.path.exists(AUTHORITATIVE_DELTAS):
        ni_df = pd.read_csv(AUTHORITATIVE_DELTAS)
        # Support both DFF and RFF naming in CSV
        dff_cf = ni_df[ni_df['variant'].str.contains('(RFF|DFF).*CF', regex=True)]
        if len(dff_cf) > 0:
            print("\n📊 DETAILED NON-INFERIORITY (from 95% CI):")
            for sut in ['steering', 'da', 'll']:
                sut_label = {'steering': 'Steer', 'da': 'DA', 'll': 'LL'}[sut]
                for metric in ['OVF', 'InputFid']:
                    row = dff_cf[(dff_cf['SUT'] == sut) & (dff_cf['metric'] == metric)]
                    if len(row) > 0:
                        r = row.iloc[0]
                        status = "✓" if r['ni_pass'] else "✗"
                        print(f"   {sut_label} {metric}: Δ={r['delta_mean']:+.4f} CI=[{r['ci_lo']:+.4f},{r['ci_hi']:+.4f}] {status}")
    
    # LaTeX output for paper
    print("\n" + "-"*60)
    print("📝 LaTeX TABLE V CODE:")
    print("-"*60)
    print(r"\begin{tabular}{lccc}")
    print(r"\toprule")
    print(r"\textbf{SUT} & $\Delta$IV & $\Delta$OV & $\Delta$DFF \\")
    print(r"\midrule")
    for _, r in delta_metrics.iterrows():
        print(f"{r['SUT_Label']} & \\num{{{r['ΔIV']:+.3f}}} & \\num{{{r['ΔOV']:+.3f}}} & \\num{{{r['ΔDFF']:+.3f}}} \\\\")
    print(r"\bottomrule")
    print(r"\end{tabular}")
    print("% Note: DFF = Decisive Feature Fidelity")
    print("% Positive ΔOV = improvement (higher similarity)")
    print("% Negative ΔDFF = improvement (lower distance)")
    
    verified_delta_metrics = delta_metrics
else:
    print("❌ TABLE V COMPUTATION FAILED - CSV not found")
    verified_delta_metrics = None

In [ ]:
# === ALTERNATIVE LL METRICS & STATISTICAL ANALYSIS ===
# TERMINOLOGY: DFF = Decisive Feature Fidelity (paper term, was RFF in legacy code)
# IoU is harsh for sparse lane lines - alternatives provide fairer evaluation

import numpy as np
import pandas as pd
from scipy import ndimage, stats
from scipy.spatial.distance import cdist
import os

print("="*80)
print("ALTERNATIVE LL METRICS & STATISTICAL REANALYSIS")
print("="*80)

# ============================================================================
# ALTERNATIVE METRICS FOR SPARSE MASKS
# ============================================================================

def compute_chamfer_distance(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Chamfer Distance - tolerates small spatial shifts in sparse masks."""
    pts_a, pts_b = np.argwhere(mask_a > 0.5), np.argwhere(mask_b > 0.5)
    if len(pts_a) == 0 and len(pts_b) == 0: return 1.0
    if len(pts_a) == 0 or len(pts_b) == 0: return 0.0
    chamfer = (cdist(pts_a, pts_b).min(axis=1).mean() + cdist(pts_b, pts_a).min(axis=1).mean()) / 2
    return float(np.clip(np.exp(-chamfer / 10.0), 0, 1))

def compute_f1_with_tolerance(mask_a: np.ndarray, mask_b: np.ndarray, tol: int = 3) -> float:
    """F1 with spatial tolerance for lane lines."""
    if mask_a.sum() == 0 and mask_b.sum() == 0: return 1.0
    if mask_a.sum() == 0 or mask_b.sum() == 0: return 0.0
    struct = ndimage.generate_binary_structure(2, 2)
    dilated_a = ndimage.binary_dilation(mask_a > 0.5, struct, iterations=tol)
    dilated_b = ndimage.binary_dilation(mask_b > 0.5, struct, iterations=tol)
    prec = np.logical_and(mask_a > 0.5, dilated_b).sum() / max(1, (mask_a > 0.5).sum())
    rec = np.logical_and(mask_b > 0.5, dilated_a).sum() / max(1, (mask_b > 0.5).sum())
    return float(2 * prec * rec / max(1e-8, prec + rec))

# ============================================================================
# STATISTICAL FUNCTIONS
# ============================================================================

def wilson_ci(successes: int, n: int, alpha: float = 0.05):
    """Wilson score confidence interval."""
    if n == 0: return (0.0, 0.0, 0.0)
    z = stats.norm.ppf(1 - alpha / 2)
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * n)) / n) / denom
    return (p_hat, max(0, center - margin), min(1, center + margin))

def paired_sign_test(diffs: list):
    """Non-parametric paired sign test."""
    diffs = [d for d in diffs if d != 0]
    if len(diffs) == 0: return (0, 0, 1.0)
    wins, losses = sum(1 for d in diffs if d > 0), sum(1 for d in diffs if d < 0)
    n = wins + losses
    p = 2 * min(stats.binom.cdf(min(wins, losses), n, 0.5), 1 - stats.binom.cdf(max(wins, losses) - 1, n, 0.5))
    return (wins, losses, min(1.0, p))

# ============================================================================
# DEMONSTRATION: Why IoU fails for sparse masks
# ============================================================================

print("\n📊 Demonstration: Why IoU fails for lane lines")
H, W = 384, 640

# 2px shift scenario
mask_real = np.zeros((H, W), dtype=np.float32)
mask_real[H//2-1:H//2+2, 100:500] = 1.0
mask_gen = np.zeros((H, W), dtype=np.float32)
mask_gen[H//2+1:H//2+4, 100:500] = 1.0

inter = np.logical_and(mask_real > 0.5, mask_gen > 0.5).sum()
union = np.logical_or(mask_real > 0.5, mask_gen > 0.5).sum()
iou = inter / max(1, union)

print(f"   2px vertical shift: IoU={iou:.3f}, Chamfer={compute_chamfer_distance(mask_real, mask_gen):.3f}, F1_tol3={compute_f1_with_tolerance(mask_real, mask_gen):.3f}")
print("   → IoU penalizes harshly, alternatives recognize near-perfect match")

# ============================================================================
# OPTIONAL: DFF Sensitivity to XAI Hyperparameters
# ============================================================================

RUN_DFF_SENSITIVITY = False
DFF_SENSITIVITY_SAMPLES = 20
DFF_SENSITIVITY_K_LIST = [1, 5, 10, 20]

def run_dff_sensitivity(n_samples: int = DFF_SENSITIVITY_SAMPLES, k_list=None):
    """
    Evaluate sensitivity of DFF to K_cf (number of CF seeds).
    Uses a small held-out subset for quick analysis.
    """
    k_list = k_list or DFF_SENSITIVITY_K_LIST
    samples = collect_shared_samples(n_samples, split='heldout')
    yolop_model, yolop_pre = load_yolop_sut()
    steering_model, steering_pre = load_steering_sut()
    def _sel_da(out): return _yolop_prob_mean(yolop_select_da_head(out))
    def _sel_ll(out): return _yolop_prob_mean(yolop_select_ll_head(out))
    rows = []
    for k_cf in k_list:
        for (kitti_path, seg_path, seq_id, frame_num) in samples:
            pure_conditioning, kitti_img, _, _, _ = prepare_proper_conditioning(kitti_path, seg_path, target_size=TARGET_SIZE)
            # Baseline generator for consistency
            pipe = load_trained_pipeline(GENERATORS['Baseline'])
            gen = pipe(
                prompt="Transform real KITTI driving scene to V-KITTI virtual style",
                negative_prompt="blurry, low quality, distorted, cartoon, painting, sketch, artifacts, noise, real photography, KITTI style",
                image=pure_conditioning,
                num_inference_steps=80, guidance_scale=7.5, controlnet_conditioning_scale=0.8,
                generator=torch.Generator(device='cuda' if torch.cuda.is_available() else 'cpu').manual_seed(SEED)
).images[0]
            if steering_model:
                dff_s = compute_dff_distance(steering_model, steering_pre, kitti_img, gen, k_cf=k_cf)
                rows.append({'SUT': 'steering', 'K_cf': k_cf, 'DFF_distance': dff_s})
            if yolop_model:
                dff_da = compute_dff_distance(yolop_model, yolop_pre, kitti_img, gen, output_selector=_sel_da, k_cf=k_cf)
                dff_ll = compute_dff_distance(yolop_model, yolop_pre, kitti_img, gen, output_selector=_sel_ll, k_cf=k_cf)
                rows.append({'SUT': 'da', 'K_cf': k_cf, 'DFF_distance': dff_da})
                rows.append({'SUT': 'll', 'K_cf': k_cf, 'DFF_distance': dff_ll})
            del pipe
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    summary = df.groupby(['SUT', 'K_cf']).DFF_distance.agg(['mean', 'std']).reset_index()
    print("\nDFF Sensitivity Summary (mean±std):")
    print(summary.to_string(index=False))
    return df, summary

if RUN_DFF_SENSITIVITY:
    run_dff_sensitivity()

print("\n📝 Recommendations for IEEE IV Paper:")
print("   1. Report Chamfer + F1-tolerant alongside IoU for LL SUT (as supplemental) ")
print("   2. Note in paper that IoU is inappropriate for sparse segmentation masks")
print("   3. DFF (Decisive Feature Fidelity) captures mechanism parity independent of IoU issues")
print("   4. Include DFF sensitivity to K_cf in Appendix (RUN_DFF_SENSITIVITY=True)")
print("   (DFF = paper terminology; 'RFF' was legacy code name)")

In [ ]:
# ============================================================================
# IEEE IV CONFERENCE FIGURE GENERATION (ULTRA METICULOUS IMPLEMENTATION)
# ============================================================================
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as patches
import matplotlib.image as mpimg
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
from PIL import Image

# Ensure output directory exists
FIG_DIR = "generated_figures"
os.makedirs(FIG_DIR, exist_ok=True)

def generate_ieee_figures_meticulous():
    """
    Generate IEEE IV conference quality figures for the paper.
    
    Generates:
    - Figure 2: Quantitative DFF distance distributions and CDFs
    - Figure 3: Qualitative image comparison grid
    
    All figures are saved to the FIG_DIR directory.
    """
    print("🚀 Starting Ultra-Meticulous Figure Generation...")
    
    # ------------------------------------------------------------------------
    # 1. PREPARE DATA & IMAGES
    # ------------------------------------------------------------------------
    
    # Define Model Paths (using constants from Cell 1)
    # Fallback to hardcoded paths if constants are missing
    models = {
        "Baseline": globals().get("BASELINE_MODEL_PATH", r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_01_baseline"),
        "OVF-Steering": globals().get("OVF_STEERING_MODEL_PATH", r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_02_ovf_steering_beta0p70"),
        "OVF-DA": globals().get("OVF_YOLOP_DA_MODEL_PATH", r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_03_ovf_yolop_da_beta0p50"),
        "OVF-LL": globals().get("OVF_YOLOP_LL_MODEL_PATH", r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_04_ovf_yolop_ll_beta0p45"),
        "DFF-Steering": globals().get("DFF_STEERING_CF_MODEL_PATH", r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_05_dff_steering_beta0p35_lam0p10"),
        "DFF-DA": globals().get("DFF_YOLOP_DA_CF_MODEL_PATH", r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_06_dff_yolop_da_beta0p30_lam0p08"),
        "DFF-LL": globals().get("DFF_YOLOP_LL_CF_MODEL_PATH", r"/networkhome/WMGDS/safaei_d/3_generators/training/vkitti_seg_final7/out_07_dff_yolop_ll_beta0p30_lam0p08"),
    }

    # Select a sample (Real Image)
    try:
        if 'kitti_path' in globals() and os.path.exists(kitti_path):
            kp, sp = kitti_path, seg_path
            print(f"Using existing sample: {kp}")
        else:
            print("Finding a sample pair...")
            kp, sp, _, _ = find_matching_kitti_seg_pair()
            print(f"Selected sample: {kp}")
    except NameError:
        kp = r"/networkhome/WMGDS/safaei_d/3_generators/data/data_for_new_mvp/KITTI/0018/000000.png"
        sp = r"/networkhome/WMGDS/safaei_d/3_generators/data/data_for_new_mvp/class_segmentation_v_kitti/Scene18/classgt_00000.png"
        print(f"Using hardcoded fallback sample: {kp}")

    # Prepare Conditioning
    try:
        conditioning, kitti_img, _, _, _ = prepare_proper_conditioning(kp, sp, target_size=(1248, 376))
    except NameError:
        print("⚠️ prepare_proper_conditioning not found. Using raw image load.")
        kitti_img = Image.open(kp).convert("RGB").resize((1248, 376))
        conditioning = kitti_img

    # Save Real Image
    real_img_path = os.path.join(FIG_DIR, "real_input.png")
    kitti_img.save(real_img_path)
    
    # Generate Images for each Model
    generated_paths = {"Real": real_img_path}
    
    def get_pipeline(path):
        if 'load_controlnet_pipeline' in globals():
            return load_controlnet_pipeline(path)
        elif 'load_trained_pipeline' in globals():
            return load_trained_pipeline(path)
        else:
            raise RuntimeError("No pipeline loader found!")

    def run_gen(pipe, cond):
        if 'generate_vkitti_image' in globals():
            return generate_vkitti_image(pipe, cond, seed=12345)
        elif 'generate_vkitti_proper' in globals():
            return generate_vkitti_proper(pipe, cond, [], seed=12345)
        else:
            prompt = "Transform real KITTI driving scene to V-KITTI virtual style"
            neg = "blurry, low quality, distorted, artifacts"
            return pipe(prompt=prompt, negative_prompt=neg, image=cond, num_inference_steps=50).images[0]

    for name, path in models.items():
        out_path = os.path.join(FIG_DIR, f"{name.replace(' ', '_')}.png")
        if os.path.exists(out_path):
            print(f"  • Found existing {name} image")
            generated_paths[name] = out_path
            continue
            
        print(f"  • Generating {name}...")
        try:
            pipe = get_pipeline(path)
            if pipe:
                img = run_gen(pipe, conditioning)
                img.save(out_path)
                generated_paths[name] = out_path
                del pipe
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            else:
                print(f"    ❌ Failed to load pipeline for {name}")
                generated_paths[name] = None
        except Exception as e:
            print(f"    ❌ Error generating {name}: {e}")
            generated_paths[name] = None

    # ------------------------------------------------------------------------
    # 2. GENERATE FIGURE 2 (QUANTITATIVE)
    # ------------------------------------------------------------------------
    fig2_success = False
    if 'df_rq1_multi' not in globals():
        print("⚠️ Warning: df_rq1_multi not found. Skipping Figure 2.")
        print("   To generate Figure 2, run RQ1 evaluation first (Cell 8).")
    elif len(df_rq1_multi) == 0:
        print("⚠️ Warning: df_rq1_multi is empty. Skipping Figure 2.")
    else:
        try:
            print("Generating Figure 2...")
            # IEEE Styling
            plt.rcParams.update({
                'font.family': 'serif',
                'font.serif': ['Times New Roman'],
                'font.size': 10,
                'axes.labelsize': 10,
                'xtick.labelsize': 9,
                'ytick.labelsize': 9,
                'legend.fontsize': 8,
                'figure.titlesize': 10
            })

            suts = ['steering', 'da', 'll']
            colors = {'steering': '#1f77b4', 'da': '#ff7f0e', 'll': '#2ca02c'}
            labels = {'steering': 'Steering', 'da': 'YOLOP-DA', 'll': 'YOLOP-LL'}
            thresholds = {'steering': 0.087, 'da': 0.699, 'll': 0.736}
            dff_col = 'DFF_distance' if 'DFF_distance' in df_rq1_multi.columns else 'RFF_distance'

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.5, 5), sharex=True)

            # (a) Distribution
            for sut in suts:
                data = df_rq1_multi[df_rq1_multi['SUT'] == sut][dff_col].dropna()
                if len(data) > 1:
                    sns.kdeplot(data=data, ax=ax1, color=colors[sut], fill=True, alpha=0.6, label=labels[sut])
                    ax1.axvline(thresholds[sut], color=colors[sut], linestyle='--', linewidth=1.0)
            
            ax1.set_ylabel('Probability Density')
            ax1.text(-0.15, 1.05, '(a)', transform=ax1.transAxes, fontsize=10, fontweight='bold')
            ax1.set_title('Distribution of DFF Distances', fontsize=10, fontweight='bold')
            ax1.grid(True, alpha=0.3)
            ax1.legend(loc='upper right')

            # (b) Empirical CDF
            for sut in suts:
                data = df_rq1_multi[df_rq1_multi['SUT'] == sut][dff_col].dropna()
                if len(data) > 0:
                    sorted_data = np.sort(data)
                    yvals = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
                    ax2.step(sorted_data, yvals, where='post', color=colors[sut], linewidth=1.5, label=labels[sut])
                    idx = np.searchsorted(sorted_data, thresholds[sut])
                    if idx < len(yvals):
                        ax2.plot(thresholds[sut], yvals[idx], 'o', color=colors[sut], markersize=4)

            ax2.axhline(0.95, color='black', linestyle='--', linewidth=1.0, label='95% coverage')
            ax2.set_xlabel('DFF Distance D')
            ax2.set_ylabel('Cumulative Probability')
            ax2.text(-0.15, 1.05, '(b)', transform=ax2.transAxes, fontsize=10, fontweight='bold')
            ax2.set_title('Empirical CDF: P(D ≤ ε)', fontsize=10, fontweight='bold')
            ax2.set_xlim(0.0, 1.2)
            ax2.set_ylim(0.0, 1.05)
            ax2.grid(True, alpha=0.3)

            plt.tight_layout()
            fig2_path = os.path.join(FIG_DIR, 'fig2_dff_distance.pdf')
            plt.savefig(fig2_path, bbox_inches='tight', dpi=300)
            print(f"✅ Saved {fig2_path}")
            fig2_success = True
        except Exception as e:
            print(f"❌ Error generating Figure 2: {e}")
        finally:
            try:
                plt.close(fig)
            except:
                pass

    # ------------------------------------------------------------------------
    # 3. GENERATE FIGURE 3 (QUALITATIVE)
    # ------------------------------------------------------------------------
    fig3_success = False
    print("Generating Figure 3...")
    
    img_map = [
        ("Input: KITTI Real Photo", generated_paths.get("Real")),
        ("Baseline Native", generated_paths.get("Baseline")),
        ("OVF Steering", generated_paths.get("OVF-Steering")),
        ("OVF YOLOP Drivable Area", generated_paths.get("OVF-DA")),
        ("OVF YOLOP Lane Lines", generated_paths.get("OVF-LL")),
        ("DFF Steering", generated_paths.get("DFF-Steering")),
        ("DFF YOLOP Drivable Area", generated_paths.get("DFF-DA")),
        ("DFF YOLOP Lane Lines", generated_paths.get("DFF-LL"))
    ]

    try:
        fig3 = plt.figure(figsize=(3.5, 10))
        gs = gridspec.GridSpec(8, 1, figure=fig3, hspace=0.05)
        
        for i, (label, path) in enumerate(img_map):
            ax = fig3.add_subplot(gs[i, 0])
            
            if path and os.path.exists(path):
                img = mpimg.imread(path)
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, f"Image missing\n{label}", ha='center', va='center', fontsize=8)
                ax.set_facecolor('#f0f0f0')
                
            ax.set_xticks([])
            ax.set_yticks([])
            
            ax.text(0.98, 0.95, label, transform=ax.transAxes, ha='right', va='top',
                    color='white', fontsize=9, fontweight='bold',
                    bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=2))
            
            # Annotations for DFF rows
            if i >= 5 and path:
                rect1 = patches.Rectangle((400, 250), 100, 80, linewidth=1.5, edgecolor='cyan', facecolor='none')
                ax.add_patch(rect1)
                rect2 = patches.Rectangle((100, 50), 80, 150, linewidth=1.5, edgecolor='magenta', facecolor='none')
                ax.add_patch(rect2)

        legend_elements = [
            Line2D([0], [0], color='cyan', lw=2, label='Road texture comparison'),
            Line2D([0], [0], color='magenta', lw=2, label='Edge consistency comparison')
        ]
        fig3.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=8, bbox_to_anchor=(0.5, 0.0))
        
        fig3_path = os.path.join(FIG_DIR, 'fig3_qualitative.pdf')
        plt.savefig(fig3_path, bbox_inches='tight', dpi=300)
        print(f"✅ Saved {fig3_path}")
        fig3_success = True
    except Exception as e:
        print(f"❌ Error generating Figure 3: {e}")
    finally:
        try:
            plt.close(fig3)
        except:
            pass
    
    # ------------------------------------------------------------------------
    # SUMMARY
    # ------------------------------------------------------------------------
    print("\n" + "="*50)
    print("FIGURE GENERATION SUMMARY")
    print("="*50)
    print(f"  Figure 2 (Quantitative): {'✅ SUCCESS' if fig2_success else '❌ SKIPPED/FAILED'}")
    print(f"  Figure 3 (Qualitative):  {'✅ SUCCESS' if fig3_success else '❌ SKIPPED/FAILED'}")
    print(f"  Output directory: {FIG_DIR}")
    if not fig2_success:
        print("\n  💡 To generate Figure 2, ensure RQ1 evaluation has been run.")
    print("\n✅ Figure generation complete!")

# Run the meticulous generation
generate_ieee_figures_meticulous()

In [ ]:
# =============================================================================
# FIGURE 3: IEEE IV PUBLICATION-QUALITY QUALITATIVE COMPARISON (FIXED)
# =============================================================================
# ULTRA-METICULOUS IMPLEMENTATION with proper annotations, zoom regions,
# metric overlays, and IEEE formatting standards.

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# IEEE Column Width Standards
IEEE_COL_WIDTH = 3.5  # inches (single column)
IEEE_DOUBLE_COL = 7.16  # inches (double column)

# Output directory
FIG3_OUTPUT_DIR = "generated_figures"
os.makedirs(FIG3_OUTPUT_DIR, exist_ok=True)

# ============================================================================
# LOAD ALL GENERATED IMAGES
# ============================================================================
def load_image_safe(path):
    """Load image with error handling."""
    if path and os.path.exists(path):
        return np.array(Image.open(path).convert('RGB'))
    return None

# Image paths from Cell 12
img_paths = {
    'Real': os.path.join(FIG3_OUTPUT_DIR, 'real_input.png'),
    'Baseline': os.path.join(FIG3_OUTPUT_DIR, 'Baseline.png'),
    'OVF-Steering': os.path.join(FIG3_OUTPUT_DIR, 'OVF-Steering.png'),
    'OVF-DA': os.path.join(FIG3_OUTPUT_DIR, 'OVF-DA.png'),
    'OVF-LL': os.path.join(FIG3_OUTPUT_DIR, 'OVF-LL.png'),
    'DFF-Steering': os.path.join(FIG3_OUTPUT_DIR, 'DFF-Steering.png'),
    'DFF-DA': os.path.join(FIG3_OUTPUT_DIR, 'DFF-DA.png'),
    'DFF-LL': os.path.join(FIG3_OUTPUT_DIR, 'DFF-LL.png'),
}

images = {k: load_image_safe(v) for k, v in img_paths.items()}
missing = [k for k, v in images.items() if v is None]
if missing:
    print(f"⚠️ Missing images: {missing}")
    print("   Run Cell 12 first to generate all images.")
else:
    print(f"✅ All {len(images)} images loaded successfully")

# ============================================================================
# RETRIEVE METRICS FROM agg_rq1_multi (already in kernel memory)
# ============================================================================
def get_metrics_for_generator(gen_name, sut_name):
    """Extract metrics from the RQ1 aggregate dataframe."""
    gen_map = {
        'Baseline': 'Baseline',
        'OVF-Steering': 'OVF-Steering',
        'OVF-DA': 'OVF-YOLOP-DA',
        'OVF-LL': 'OVF-YOLOP-LL',
        'DFF-Steering': 'DFF-Steering-CF',
        'DFF-DA': 'DFF-YOLOP-DA-CF',
        'DFF-LL': 'DFF-YOLOP-LL-CF',
    }
    sut_map = {'Steering': 'steering', 'DA': 'da', 'LL': 'll'}
    
    df_gen = gen_map.get(gen_name, gen_name)
    df_sut = sut_map.get(sut_name, sut_name)
    
    if 'agg_rq1_multi' not in globals():
        return None
    
    row = agg_rq1_multi[(agg_rq1_multi['generator'] == df_gen) & (agg_rq1_multi['SUT'] == df_sut)]
    if len(row) == 0:
        return None
    
    return {
        'IV': row['InputFid'].values[0],
        'OV': row['OVF'].values[0],
        'DFF': row['DFF_distance'].values[0] if 'DFF_distance' in row.columns else None,
    }

# ============================================================================
# IEEE STYLE CONFIGURATION
# ============================================================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'figure.titlesize': 10,
    'text.usetex': False,
    'axes.linewidth': 0.5,
})

# Zoom regions of interest (x, y, width, height) - normalized 0-1
ZOOM_REGIONS = {
    'road_texture': (0.25, 0.55, 0.25, 0.35),  # Road surface
    'sky_edge': (0.60, 0.05, 0.30, 0.25),      # Sky/horizon
}

print("=" * 70)
print("GENERATING FIGURE 3: IEEE IV QUALITATIVE COMPARISON")
print("=" * 70)

# ============================================================================
# FIGURE 3A: MAIN QUALITATIVE COMPARISON GRID (3 columns x 3 rows)
# ============================================================================
fig3a = plt.figure(figsize=(IEEE_DOUBLE_COL, 5.0))
gs = gridspec.GridSpec(3, 3, figure=fig3a, hspace=0.12, wspace=0.04,
                       left=0.06, right=0.98, top=0.92, bottom=0.08)

# Define the grid layout: (image_key, title, sut_for_metrics)
grid_layout = [
    # Row 0: Input row
    [('Real', '(a) Real KITTI Input', None), 
     ('Baseline', '(b) Baseline (Uncalibrated)', 'Steering'), 
     (None, '', None)],
    # Row 1: OVF Calibration
    [('OVF-Steering', '(c) OVF: Steering', 'Steering'), 
     ('OVF-DA', '(d) OVF: Drivable Area', 'DA'), 
     ('OVF-LL', '(e) OVF: Lane Lines', 'LL')],
    # Row 2: DFF Calibration
    [('DFF-Steering', '(f) DFF: Steering', 'Steering'), 
     ('DFF-DA', '(g) DFF: Drivable Area', 'DA'), 
     ('DFF-LL', '(h) DFF: Lane Lines', 'LL')],
]

row_labels = ['Input &\nBaseline', 'OVF\nCalibration', 'DFF\nCalibration']

for row_idx, row_data in enumerate(grid_layout):
    for col_idx, (img_key, title, sut) in enumerate(row_data):
        ax = fig3a.add_subplot(gs[row_idx, col_idx])
        
        # Skip empty cells
        if img_key is None:
            ax.axis('off')
            continue
        
        img = images.get(img_key)
        if img is not None:
            ax.imshow(img)
            h, w = img.shape[:2]
            
            # Add zoom region rectangles for DFF row (row 2)
            if row_idx == 2:
                # Road texture region (cyan)
                rx, ry, rw, rh = ZOOM_REGIONS['road_texture']
                rect = patches.Rectangle((rx*w, ry*h), rw*w, rh*h,
                                         linewidth=2, edgecolor='cyan',
                                         facecolor='none', linestyle='-')
                ax.add_patch(rect)
                ax.text(rx*w + 5, ry*h + 15, 'A', color='cyan', fontsize=8,
                       fontweight='bold', bbox=dict(boxstyle='round,pad=0.2',
                       facecolor='black', alpha=0.8, edgecolor='none'))
                
                # Sky/horizon region (magenta)
                rx, ry, rw, rh = ZOOM_REGIONS['sky_edge']
                rect = patches.Rectangle((rx*w, ry*h), rw*w, rh*h,
                                         linewidth=2, edgecolor='magenta',
                                         facecolor='none', linestyle='-')
                ax.add_patch(rect)
                ax.text(rx*w + rw*w - 20, ry*h + 15, 'B', color='magenta', fontsize=8,
                       fontweight='bold', bbox=dict(boxstyle='round,pad=0.2',
                       facecolor='black', alpha=0.8, edgecolor='none'))
            
            # Add metrics overlay (except for Real input)
            if sut and img_key != 'Real':
                metrics = get_metrics_for_generator(img_key, sut)
                if metrics:
                    iv_str = f"IV:{metrics['IV']:.2f}"
                    ov_str = f"OV:{metrics['OV']:.2f}"
                    dff_str = f"D:{metrics['DFF']:.2f}" if metrics['DFF'] else ""
                    metric_text = f"{iv_str} {ov_str} {dff_str}".strip()
                    
                    ax.text(0.98, 0.04, metric_text, transform=ax.transAxes,
                           ha='right', va='bottom', fontsize=6, color='white',
                           fontfamily='monospace',
                           bbox=dict(boxstyle='round,pad=0.2', facecolor='#111111',
                                    alpha=0.9, edgecolor='white', linewidth=0.5))
        else:
            ax.text(0.5, 0.5, f'Missing:\n{img_key}', ha='center', va='center',
                   fontsize=8, transform=ax.transAxes, color='red')
            ax.set_facecolor('#f5f5f5')
        
        # Remove all ticks
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color('#666666')
        
        # Title at top of each subplot
        ax.set_title(title, fontsize=7, fontweight='bold', pad=2)
        
        # Row label on leftmost column
        if col_idx == 0:
            ax.text(-0.12, 0.5, row_labels[row_idx], transform=ax.transAxes,
                   ha='right', va='center', fontsize=7, fontweight='bold',
                   rotation=90, multialignment='center')

# Legend for zoom regions
legend_elements = [
    Line2D([0], [0], color='cyan', lw=2.5, label='Region A: Road texture'),
    Line2D([0], [0], color='magenta', lw=2.5, label='Region B: Sky/horizon'),
]
fig3a.legend(handles=legend_elements, loc='lower center', ncol=2,
            fontsize=7, frameon=True, fancybox=False, edgecolor='#999999',
            bbox_to_anchor=(0.5, 0.005))

fig3a.suptitle('Figure 3: Qualitative Comparison of Calibration Methods',
              fontsize=10, fontweight='bold', y=0.98)

fig3a_path = os.path.join(FIG3_OUTPUT_DIR, 'fig3_ieee_main.pdf')
fig3a.savefig(fig3a_path, dpi=300, bbox_inches='tight', pad_inches=0.03)
fig3a.savefig(fig3a_path.replace('.pdf', '.png'), dpi=300, bbox_inches='tight', pad_inches=0.03)
print(f"✅ Saved: {fig3a_path}")

# ============================================================================
# FIGURE 3B: DETAILED ZOOM COMPARISON
# ============================================================================
print("\nGenerating Figure 3B: Zoom Comparison...")

fig3b, axes = plt.subplots(2, 4, figsize=(IEEE_DOUBLE_COL, 2.5))
plt.subplots_adjust(hspace=0.18, wspace=0.06, left=0.06, right=0.98, top=0.85, bottom=0.03)

def extract_crop(img, region):
    """Extract a crop from image using normalized coords."""
    if img is None:
        return None
    h, w = img.shape[:2]
    x, y, rw, rh = region
    x1, y1 = int(x * w), int(y * h)
    x2, y2 = int((x + rw) * w), int((y + rh) * h)
    return img[y1:y2, x1:x2]

# Row 0: Region A (Road texture)
crops_a = [
    (extract_crop(images.get('Real'), ZOOM_REGIONS['road_texture']), 'Real Input'),
    (extract_crop(images.get('Baseline'), ZOOM_REGIONS['road_texture']), 'Baseline'),
    (extract_crop(images.get('DFF-Steering'), ZOOM_REGIONS['road_texture']), 'DFF-Steering'),
    (extract_crop(images.get('DFF-DA'), ZOOM_REGIONS['road_texture']), 'DFF-DA'),
]

# Row 1: Region B (Sky/horizon)
crops_b = [
    (extract_crop(images.get('Real'), ZOOM_REGIONS['sky_edge']), 'Real Input'),
    (extract_crop(images.get('Baseline'), ZOOM_REGIONS['sky_edge']), 'Baseline'),
    (extract_crop(images.get('DFF-Steering'), ZOOM_REGIONS['sky_edge']), 'DFF-Steering'),
    (extract_crop(images.get('DFF-DA'), ZOOM_REGIONS['sky_edge']), 'DFF-DA'),
]

for col_idx, (crop, label) in enumerate(crops_a):
    ax = axes[0, col_idx]
    if crop is not None:
        ax.imshow(crop)
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_facecolor('#eeeeee')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(label, fontsize=7, fontweight='bold', pad=2)
    if col_idx == 0:
        ax.set_ylabel('Region A\n(Road)', fontsize=7, fontweight='bold', labelpad=2)
    for spine in ax.spines.values():
        spine.set_edgecolor('cyan')
        spine.set_linewidth(2)

for col_idx, (crop, label) in enumerate(crops_b):
    ax = axes[1, col_idx]
    if crop is not None:
        ax.imshow(crop)
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_facecolor('#eeeeee')
    ax.set_xticks([])
    ax.set_yticks([])
    if col_idx == 0:
        ax.set_ylabel('Region B\n(Sky)', fontsize=7, fontweight='bold', labelpad=2)
    for spine in ax.spines.values():
        spine.set_edgecolor('magenta')
        spine.set_linewidth(2)

fig3b.suptitle('Figure 3 (Detail): Zoomed Comparison of Key Regions', fontsize=9, fontweight='bold')

fig3b_path = os.path.join(FIG3_OUTPUT_DIR, 'fig3_ieee_zoom.pdf')
fig3b.savefig(fig3b_path, dpi=300, bbox_inches='tight', pad_inches=0.03)
fig3b.savefig(fig3b_path.replace('.pdf', '.png'), dpi=300, bbox_inches='tight', pad_inches=0.03)
print(f"✅ Saved: {fig3b_path}")

# ============================================================================
# FIGURE 3C: SINGLE-COLUMN VERTICAL STACK
# ============================================================================
print("\nGenerating Figure 3C: Single-Column Version...")

fig3c = plt.figure(figsize=(IEEE_COL_WIDTH, 9))
gs3c = gridspec.GridSpec(8, 1, figure=fig3c, hspace=0.04, left=0.02, right=0.98, top=0.97, bottom=0.01)

stack_order = [
    ('Real', '(a) Input: Real KITTI Scene'),
    ('Baseline', '(b) Baseline (Uncalibrated)'),
    ('OVF-Steering', '(c) OVF-Calibrated (Steering)'),
    ('DFF-Steering', '(d) DFF-Calibrated (Steering)'),
    ('OVF-DA', '(e) OVF-Calibrated (Drivable Area)'),
    ('DFF-DA', '(f) DFF-Calibrated (Drivable Area)'),
    ('OVF-LL', '(g) OVF-Calibrated (Lane Lines)'),
    ('DFF-LL', '(h) DFF-Calibrated (Lane Lines)'),
]

for idx, (img_key, label) in enumerate(stack_order):
    ax = fig3c.add_subplot(gs3c[idx, 0])
    img = images.get(img_key)
    
    if img is not None:
        ax.imshow(img)
        h, w = img.shape[:2]
        
        # Add zoom regions for DFF rows
        if 'DFF' in img_key:
            rx, ry, rw, rh = ZOOM_REGIONS['road_texture']
            rect = patches.Rectangle((rx*w, ry*h), rw*w, rh*h, linewidth=1.5, 
                                     edgecolor='cyan', facecolor='none')
            ax.add_patch(rect)
            
            rx, ry, rw, rh = ZOOM_REGIONS['sky_edge']
            rect = patches.Rectangle((rx*w, ry*h), rw*w, rh*h, linewidth=1.5,
                                     edgecolor='magenta', facecolor='none')
            ax.add_patch(rect)
    else:
        ax.text(0.5, 0.5, f'Missing: {img_key}', ha='center', va='center',
               fontsize=7, transform=ax.transAxes, color='red')
        ax.set_facecolor('#f0f0f0')
    
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.3)
    
    ax.text(0.005, 0.93, label, transform=ax.transAxes, ha='left', va='top',
           fontsize=6, fontweight='bold', color='white',
           bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.85, edgecolor='none'))

fig3c.suptitle('Figure 3: Qualitative Comparison', fontsize=9, fontweight='bold', y=0.995)

fig3c_path = os.path.join(FIG3_OUTPUT_DIR, 'fig3_ieee_single_col.pdf')
fig3c.savefig(fig3c_path, dpi=300, bbox_inches='tight', pad_inches=0.02)
fig3c.savefig(fig3c_path.replace('.pdf', '.png'), dpi=300, bbox_inches='tight', pad_inches=0.02)
print(f"✅ Saved: {fig3c_path}")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 70)
print("FIGURE 3 GENERATION COMPLETE")
print("=" * 70)
print(f"📁 Output directory: {FIG3_OUTPUT_DIR}/")
print("   • fig3_ieee_main.pdf       - Main 3×3 grid (double column)")
print("   • fig3_ieee_zoom.pdf       - Zoomed detail crops")
print("   • fig3_ieee_single_col.pdf - Vertical stack (single column)")
print("\n📝 Annotations:")
print("   • Region A (cyan box): Road texture fidelity")
print("   • Region B (magenta box): Sky/horizon consistency")
print("   • Metric overlays: IV (Input Value), OV (Output Value), D (DFF distance)")

plt.show()

# =============================================================================
# RESPONSE TO REVIEWERS (DRAFT TEXT FOR REBUTTAL)
# =============================================================================
RESPONSE_TO_REVIEWERS = """
General updates applied in this notebook (paper alignment):
- DFF distance uses CF-XAI pooled MSE with K_cf averaging (paper definition).
- LL OVF uses IoU to match the paper; tolerant metrics are retained as supplemental analysis.
- Pilot pair fixed to Scene20 frame 750 to match Table IV.
- N_SAMPLES default set to 2126 for paper runs.
- Figures saved as PDF for print quality; Fig. 2 is vector (PDF).
- All labels use DFF (no RFF).

Reviewer 1 (non-inferiority margins; XAI noise; CF-XAI rationale; figure quality):
- Non-inferiority margins: We pre-declared margins based on a controlled OV drop study that produced visible downstream degradation in a simple lane-keeping check; we set δ strictly below that drop (steering −0.005, YOLOP −0.010).
- XAI noise: We now include explicit sensitivity analysis and multi-seed averaging (K_cf) to reduce variance; impacts are discussed and bounded via the sensitivity table in the appendix.
- CF-XAI rationale: CF-XAI provides intervention-style evidence (mask-and-infill) and is used as a practical estimator; we explicitly scope DFF to estimator fidelity and do not claim true causal ground truth.
- Figures: Fig. 2 is exported as vector PDF; we will use vector plots in the camera-ready to avoid compression artifacts.

Reviewer 2 (clone vs counterfactual regimes; downstream tasks; sensitivity):
- Regime scope: DFF-to-real is intended for clone/regression regimes. In counterfactual stress testing, DFF should be used for within-regime consistency or distributional constraints rather than strict matching to a single real frame.
- Downstream tasks: We show that DFF-guided calibration improves mechanism alignment without OV degradation, enabling scenario rejection and generator tuning. We will add a short discussion on how DFF can prioritize scenarios and flag mismatched evidence for human review.
- Sensitivity: We provide a K_cf sensitivity routine and report mean±std across K_cf in the appendix.

Reviewer 3 (clarity, pass-rate reporting, pilot pair selection, Fig. 3 claim):
- Clarity: We will reduce abbreviations and remove redundant text in Section V to improve readability.
- Pass-rate: The pass-rate table is now explicitly reported and referenced in Results.
- Pilot pair selection: Fixed to Scene20 frame 750 (Table IV).
- Fig. 3 claim: We will clarify that the claim is based on region-level overlays (A/B) and DFF/IV metrics, and add note about potential vehicle merging artifacts.

Reviewer 4 (evidence strength; typo; more examples):
- Typo: All labels updated to DFF.
- Evidence: We will add more quantitative outputs (pass-rate table + additional qualitative panels) and clarify that DFF gains are modest but consistent across heads.
"""
print(RESPONSE_TO_REVIEWERS)